<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T04-L01-grpo-from-scratch/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T04-L01-grpo-from-scratch/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T04-L01-grpo-from-scratch/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T04-L01-grpo-from-scratch/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T04-L01 · GRPO from scratch on a tiny model

**You will build:** Group Relative Policy Optimisation, the reinforcement-learning recipe behind
DeepSeek-R1-Zero, in plain numpy on a policy small enough to train in seconds: hand-derived
gradients checked by finite differences, group sampling, a program that verifies the answer, group-relative
advantages, the clipped surrogate objective, the KL penalty, a training loop, and then the measured traps.

**Time:** ~60 minutes · **Runs on:** a laptop CPU, no GPU, no download ·
**Prerequisites:** T00-L01-the-8gb-track.

This is a TOY. The policy has parameters you can count on the page (section 2 prints them), not billions; the task is arithmetic a
program can check; the model is handed a scratchpad feature a real model would have to learn. What it
does and does not show about real reasoning models is spelled out in section 21, and every claim about
the world in this file has a dated source in `claims.yaml`.

By the end you will be able to:

1. Implement a small autoregressive softmax policy with a hand-derived backward pass, and prove the
   gradient right against central finite differences.
2. Implement group sampling, a program-checked verifier with a separate format reward, and the
   group-relative advantage, including its epsilon and its zero-variance case.
3. Implement the clipped surrogate objective, the k3 KL estimator, and three sequence-length
   normalisations, and assemble them into a GRPO update over several epochs per batch.
4. Measure, over a fixed set of seeds with intervals, reward against step, how often a group carries zero
   gradient, and what removing the standard-deviation normalisation, changing the length
   normalisation, or changing the group size does on this task.
5. Measure a reward hack against a verifier with a loophole, and explain what this toy does not show.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T00-L01`

**This lesson assumes you can already:**

- write Python functions over numpy arrays and index them with integer arrays and boolean masks
- differentiate a composition of functions with the chain rule, and say what softmax and the log of a softmax output are
- say what a probability distribution over next tokens is, and draw a sample from one with numpy
- compute a mean and a standard deviation, and say what a 95% interval around a mean over several runs claims

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Spinning Up in Deep RL: Part 3, Intro to Policy Optimization](https://spinningup.openai.com/en/latest/spinningup/rl_intro3.html) · OpenAI (Joshua Achiam) · *free* — derives the simplest policy gradient, the expected grad-log-prob lemma and why a baseline leaves the gradient unbiased, the idea GRPO's group mean puts to use
- [Policy Gradient Algorithms](https://lilianweng.github.io/posts/2018-04-08-policy-gradient/) · Lilian Weng · *free* — REINFORCE, advantages and baselines, then PPO's clipped objective, which is the surrogate the lesson clips
- [Dive into Deep Learning, section 4.1 Softmax Regression](https://d2l.ai/chapter_linear-classification/softmax-regression.html) · Zhang, Lipton, Li and Smola · *free* — derives the gradient of the softmax cross-entropy loss with respect to the logits (softmax minus one-hot), the building block of the hand-derived backward pass
- [DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models](https://arxiv.org/abs/2402.03300) · Shao et al. (arXiv) · *free* — section 4.1 introduces GRPO: the group-relative advantage, the clipped objective and the KL penalty that the exercises implement
- [OpenIntro Statistics](https://www.openintro.org/book/os/) · Diez, Cetinkaya-Rundel and Barr · *free* — chapter 7 covers the t-distribution and confidence intervals for a mean, the Student-t interval the lesson puts around a mean over seeds

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import sys
import time
import traceback
from typing import Any, Callable, NamedTuple

import numpy as np

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__, "· platform", sys.platform)

SEED = 20260929          # this lesson's one fixed seed. Every initial policy, every sample and every
                          # seed-sweep below derives from it, so re-running prints the same numbers.

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("policy_logits",),
    "exercise 2": ("logprob_grad",),
    "exercise 3": ("sample_group",),
    "exercise 4": ("check_completion",),
    "exercise 5": ("group_advantages", "zero_variance_fraction"),
    "exercise 6": ("clipped_surrogate",),
    "exercise 7": ("k3_kl",),
    "exercise 8": ("length_scale",),
    "exercise 9": ("grpo_update",),
    "exercise 10": ("train_grpo",),
}


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (sample_group)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


# --- COMMONS TIME LIMIT v4 --- every check in this notebook runs under a time limit
import contextlib, gc, os, signal, subprocess, sys, threading, time


# A time limit for every check that calls the student's code: a loop that never ends must fail
# its check, not freeze the notebook or the grader (which imports the notebook). Copy this block
# unchanged; change only the three limits below it. Why it is this careful:
#  * Whoever runs the notebook may already have an alarm running (tools/grade.py stops every
#    rubric row after 45 s; a harness may have one whose default action ends the process). This
#    guard saves it, lets whichever deadline comes first win (handing the signal to the outer
#    handler when it is the outer's turn) and on the way out re-arms it with the time it had left
#    and its interval. Switching it off instead would let the NEXT hang run for ever.
#  * Its own alarm is one-shot, re-armed by hand every 0.25 s once the time is up (in case a bare
#    `except:` swallows the first). A second alarm alone is not enough: `while True: try: ...
#    except: pass` is interrupted inside the `try` every time and carries on. So from the second
#    stop on, a line tracer (sys.settrace, on the running frames too) raises the stop again at the
#    first line the code runs afterwards, which is in the handler, outside the `try`; it is put
#    back as it was on the way out. A repeating alarm kept firing while a runaway loop's huge list
#    was being torn down; a repeat that arrived after the default handler was back ended the
#    process (exit -14, a dead Jupyter kernel), and one that arrived inside contextlib's exit
#    escaped as a raw timeout and left this handler installed.
#    On Python 3.12+ that tracer is sys.monitoring, which stays on after it raises, so a loop
#    inside TWO bare `except:` blocks is stopped as well: the Nth stop lets it raise N-1 times in
#    a row (one per handler it has to leave), and no more, so the `finally:` blocks and context
#    managers that run as the stop travels out are not interrupted again. On 3.11 sys.settrace is
#    unset by the interpreter each time it raises, so only the innermost one is peeled per alarm.
#  * It also watches memory: every 0.1 s it compares the process's resident memory with what it
#    was when the block started, and stops the block as soon as the growth passes `memory_mib`.
#    A loop that appends to a list reaches gigabytes in seconds, long before any sensible time
#    limit, so the time limit alone does not bound it. (Resident memory NOW, not the peak so
#    far: a peak only ever rises, so after one runaway check the next would be allowed another
#    `memory_mib` on top of it, and five looping exercises would reach ten gigabytes. Where the
#    system will not say, /proc on Linux and libproc on macOS being the two that do, it falls
#    back to the peak.)
#  * It never raises inside its own set-up, exit or clean-up (nor inside another guard's: guards
#    nest, the grader's wraps this one), and on the way out it takes any alarm of its own that has
#    gone off but not yet been handled (with SIGALRM blocked) before the old handler is put back.
# tools/verify_all.py fails a lesson whose guard does less.
class _OutOfTime(BaseException):
    """Raised inside the guarded block when its time is up. A BaseException, so `except
    Exception:` in the code under test cannot swallow it; it repeats every 0.25 s, and from the
    second time on a line tracer raises it again at the next line the code runs (inside the
    handler that swallowed the first), in case a bare `except:` does. `_time_limit` turns it into
    an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 0):
    """Fail the block with an AssertionError naming `what` if it runs longer than `seconds`, or
    (when `memory_mib` is set) if the process's resident memory grows by more than that while it runs.
    Uses the operating system's alarm (Linux and macOS, main thread); a no-op elsewhere."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped check's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose stopping condition "
               "is never met runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)
    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


# The three limits. A correct answer needs a small fraction of the first two (at least five times
# less, measured on the worked solution, on the slower of a Mac and an emulated Linux machine), so
# only a loop that never stops, or one that keeps appending until memory runs out, ever meets one.
_CHECK_SECONDS = 2     # an exercise's check
_DEMO_SECONDS = 45     # any other cell that runs your code: a demo, a table, a figure
_MEMORY_MIB = 2048     # how far a check may grow the process's resident memory before it is stopped


def _limited(label: str, check):
    """Run `check()` under `_time_limit` and return what it returns: _CHECK_SECONDS for an
    exercise's check, _DEMO_SECONDS for anything else. A check that is stopped (too long, or too
    much memory) fails with an AssertionError that says so; whatever the stopped code had built (a
    runaway list can hold gigabytes) is freed at once, not whenever the garbage collector next
    runs; and every program it had started that is still running is stopped too."""
    seconds = _CHECK_SECONDS if label.lower().startswith("exercise") else _DEMO_SECONDS
    before = _processes()
    try:
        with _time_limit(seconds, "this check", _MEMORY_MIB):
            return check()
    except AssertionError as exc:
        if not str(exc).startswith("this check was "):
            raise                                        # the check's own verdict, unchanged
        message = str(exc)
    gc.collect()          # the stopped frames hold what they built in a reference cycle until now
    _stop_strays(before)
    raise AssertionError(message)


def _processes() -> dict:
    """{pid: (parent, process group)} of every process (Linux: /proc; macOS: ps); {} if unreadable."""
    table = {}
    try:
        if os.path.isdir("/proc/self"):
            for entry in os.listdir("/proc"):
                if entry.isdigit():
                    try:
                        with open(f"/proc/{entry}/stat") as f:
                            fields = f.read().rsplit(")", 1)[1].split()
                        table[int(entry)] = (int(fields[1]), int(fields[2]))
                    except (OSError, ValueError, IndexError):
                        continue
        else:
            listing = subprocess.run(["ps", "-A", "-o", "pid=,ppid=,pgid="], capture_output=True, text=True,
                                     timeout=10).stdout
            for row in listing.splitlines():
                pid, ppid, pgid = (int(x) for x in row.split())
                table[pid] = (ppid, pgid)
    except (OSError, ValueError, subprocess.SubprocessError):
        return {}
    return table


def _stop_strays(before: dict) -> None:
    """Stopping a check stops the program it was waiting for, but not what THAT program started:
    stop `make test` and the test binary make launched runs on, at full speed, for ever. Stop every
    process that did not exist before the check (`before`, from _processes()) and either descends
    from this one or sits in its process group (an orphan: its parent is gone, so it now belongs to
    pid 1 or to a subreaper, whichever this machine has). Nothing that was already running is
    touched, the launcher that started this notebook included. Best effort."""
    now, me, group = _processes(), os.getpid(), os.getpgrp()
    if not before or not now:
        return
    def descends(pid):
        seen = set()
        while pid in now and pid not in seen:
            seen.add(pid)
            pid = now[pid][0]
            if pid == me:
                return True
        return False
    for pid, (_, pgid) in now.items():
        if pid != me and pid not in before and (pgid == group or descends(pid)):
            try:
                os.kill(pid, signal.SIGKILL)
            except OSError:
                pass
# --- END COMMONS TIME LIMIT ---


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        _limited(label, check)
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


@contextlib.contextmanager
def _budget(name: str, limit: int, mistake: str, rows: bool = False):
    """Inside the block, let the lesson function `name` be called at most `limit` times (or, with
    `rows=True`, on at most `limit` decision rows in total), then stop with `mistake` as a failed
    check. A loop in your code that never reaches its stopping condition becomes a message here,
    instead of a notebook that never finishes. Correct code stays far inside every budget."""
    original, used = globals()[name], [0]

    def counted(*args, **kwargs):
        used[0] += np.size(args[1] if len(args) > 1 else kwargs.get("rem", ())) if rows else 1
        if used[0] > limit:
            raise AssertionError(mistake)
        return original(*args, **kwargs)

    globals()[name] = counted
    try:
        yield
    finally:
        globals()[name] = original

## 1. What GRPO is, in one page

Reinforcement learning from a **verifiable reward** trains a model by sampling its own answers, scoring them
with a program, and making the well-scored ones more likely. Group Relative Policy Optimisation (GRPO,
introduced in DeepSeekMath) does the scoring-to-learning step without a learned value function: for each
prompt it samples a GROUP of `G` completions and uses the group's own average reward as the baseline
(claims.yaml `grpo-no-critic`). DeepSeek-R1-Zero was trained with it, using rule-based
accuracy and format rewards (`r1-uses-grpo`, `r1-rule-based-rewards`).

For each prompt `q` and each completion `i` in its group:

1. **Reward** `r_i` from a program: is the answer right, and is the format right?
2. **Advantage** `A_i = (r_i − mean(r)) / (std(r) + ε)`, the same number for every token of completion `i`
   (`grpo-advantage-normalisation`). A group whose rewards are all equal has `A_i = 0` everywhere: it teaches
   nothing.
3. **Surrogate objective** per token: `min(ρ·A, clip(ρ, 1−ε_c, 1+ε_c)·A)` where
   `ρ = π_θ(token) / π_old(token)`. Several gradient epochs run on one batch
   (`grpo-multiple-iterations`), and the clip, inherited from PPO, is there for stability
   (`ppo-clip-stabilises`).
4. **KL penalty** `β·KL(π_θ ‖ π_ref)` added to the loss, estimated per token by the k3 estimator
   (`grpo-kl-estimator`, `k3-estimator`).
5. **Length normalisation**: the published objective averages each completion's token terms over that
   completion's own length (`dapo-sample-level-loss`). Later work argues that this, and the division by the
   standard deviation, are biases (`drgrpo-length-bias`, `drgrpo-difficulty-bias`); sections 17 and 16
   measure both on this task.

As of 2026-09-29 those are the definitions the sources give; the lesson implements exactly those, with the
variants named where they differ.

## 2. The task, the policy, and what is given to you

**The task.** A prompt is a target `T`. The model emits digit tokens `0`–`9`, then an end-of-sequence token
`EOS`, in at most `LMAX` tokens. A program checks the completion: did it end with `EOS` (the *format*), and do
its digits sum to exactly `T` (the *answer*)? The reward is a small format reward plus a larger correctness
reward, so it takes three values.

**The policy.** At every step the policy sees three things: how much of the target is still to be made
(`T` minus the digits so far, with "overshot" as its own class), the previous token, and the position. One
hidden layer of `tanh` units turns the sum of three learned embeddings into a softmax over the vocabulary.
Handing it the remaining-sum feature is a deliberate simplification: a real model would have to compute that
from the text. Here it lets the lesson study the reinforcement learning and nothing else.

The cell below is given. It defines the constants, the parameters, and the helpers you build on. Run it and
read what it prints.

In [ ]:
DIGITS, EOS, BOS = 10, 10, 11          # tokens 0-9 are digits, 10 ends a completion, 11 starts one
VOCAB = 11                             # the policy chooses among digits and EOS; it never emits BOS
N_PREV = 12                            # previous-token classes: the 11 tokens plus BOS
LMAX = 8                               # a completion has at most this many tokens, EOS included
TMAX = 16                              # the largest target
N_REM = TMAX + 2                       # remaining-sum classes: "overshot" (class 0), then 0..TMAX
HIDDEN = 16                            # tanh units
FORMAT_REWARD, CORRECT_REWARD = 0.1, 1.0
ADV_EPS = 1e-4                         # the epsilon in (r - mean) / (std + eps)
TRAIN_TARGETS = np.arange(2, TMAX + 1, 2)      # the prompts the policy trains on
HELD_OUT_TARGETS = np.arange(1, TMAX, 2)       # prompts it never sees during training
INIT_SCALE = 0.3

PARAM_NAMES = ("rem_emb", "prev_emb", "pos_emb", "b1", "W2", "b2")


def init_params(seed: int) -> dict[str, np.ndarray]:
    """A fresh policy. Small random weights: the untrained policy is close to uniform, not to right."""
    rng = np.random.default_rng([SEED, seed, 0])
    return {
        "rem_emb": rng.normal(0, INIT_SCALE, (N_REM, HIDDEN)),
        "prev_emb": rng.normal(0, INIT_SCALE, (N_PREV, HIDDEN)),
        "pos_emb": rng.normal(0, INIT_SCALE, (LMAX, HIDDEN)),
        "b1": np.zeros(HIDDEN),
        "W2": rng.normal(0, INIT_SCALE, (HIDDEN, VOCAB)),
        "b2": np.zeros(VOCAB),
    }


def copy_params(params: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    return {name: value.copy() for name, value in params.items()}


def n_parameters(params: dict[str, np.ndarray]) -> int:
    return sum(value.size for value in params.values())


def rem_class(remaining: np.ndarray) -> np.ndarray:
    """Map "how much of the target is still to be made" to an embedding row: overshot -> 0, else r + 1."""
    remaining = np.asarray(remaining)
    return np.where(remaining < 0, 0, remaining + 1)


def log_softmax(logits: np.ndarray) -> np.ndarray:
    shifted = logits - logits.max(axis=-1, keepdims=True)
    return shifted - np.log(np.exp(shifted).sum(axis=-1, keepdims=True))


def softmax(logits: np.ndarray) -> np.ndarray:
    return np.exp(log_softmax(logits))


class Rows(NamedTuple):
    """One row per token actually emitted (EOS included): the context it was chosen in, and the choice."""
    rem: np.ndarray       # remaining-sum CLASS at that step (already through rem_class)
    prev: np.ndarray      # previous token (BOS at the first step)
    pos: np.ndarray       # 0-based position
    action: np.ndarray    # the token that was emitted
    seq: np.ndarray       # which completion (row of the sample array) this token belongs to


_params0 = init_params(0)
print(f"policy: {n_parameters(_params0)} parameters in {len(_params0)} arrays "
      f"({', '.join(f'{name} {value.shape}' for name, value in _params0.items())})")
print(f"vocabulary {VOCAB} (digits 0-{DIGITS - 1} and EOS), completions of at most {LMAX} tokens, "
      f"targets 1-{TMAX}")
print(f"trained on the {len(TRAIN_TARGETS)} targets {TRAIN_TARGETS.tolist()}; held out: "
      f"{HELD_OUT_TARGETS.tolist()}")

## 3. Exercise 1 — `policy_logits()`

The forward pass. Given one row per decision — the remaining-sum class, the previous token and the position,
each an integer array of length `n` — return the logits over the vocabulary, shape `(n, VOCAB)`:

`pre = b1 + rem_emb[rem] + prev_emb[prev] + pos_emb[pos]`, then `h = tanh(pre)`, then `logits = h @ W2 + b2`.

The three embeddings are what a one-hot input times a weight matrix would give, written as a table lookup.

<details><summary>💡 Hint 1 — what to think about</summary>

Indexing a 2-D array with an integer array picks rows, so each embedding lookup is a single index
expression and already has shape `(n, HIDDEN)`. Add the three lookups and the bias `b1`, apply `tanh`, and
you have the hidden layer for all `n` rows at once, with no Python loop.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

The output layer is a matrix product of the hidden activations `(n, HIDDEN)` with `W2` `(HIDDEN, VOCAB)`,
plus `b2`. A matrix product is fine; the reference solution spells it as a broadcast multiply and a sum so
the printed numbers do not depend on which BLAS library a machine has. The result is logits, not
probabilities: no softmax here.
</details>

In [ ]:
def policy_logits(params: dict[str, np.ndarray], rem: np.ndarray, prev: np.ndarray,
                  pos: np.ndarray) -> np.ndarray:
    """Logits over the vocabulary for `n` decisions at once.

    `rem`, `prev`, `pos` are integer arrays of shape (n,): the remaining-sum CLASS (already through
    `rem_class`), the previous token, and the position. Returns an array of shape (n, VOCAB).

    Returns: the un-normalised log-probabilities, `tanh(b1 + rem_emb[rem] + prev_emb[prev] +
    pos_emb[pos]) @ W2 + b2`.

    Example:
        >>> p = init_params(0)
        >>> policy_logits(p, np.array([3, 5]), np.array([BOS, 4]), np.array([0, 1])).shape
        (2, 11)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_policy_logits() -> None:
    p = init_params(0)
    rem, prev, pos = np.array([3, 0, 17]), np.array([BOS, 4, 10]), np.array([0, 1, 7])
    out = policy_logits(p, rem, prev, pos)
    assert out.shape == (3, VOCAB), f"expected shape (3, {VOCAB}), got {out.shape}"
    hidden = np.tanh(p["b1"] + p["rem_emb"][rem[0]] + p["prev_emb"][prev[0]] + p["pos_emb"][pos[0]])
    expected = np.array([sum(hidden[k] * p["W2"][k, v] for k in range(HIDDEN)) + p["b2"][v]
                         for v in range(VOCAB)])
    assert np.allclose(out[0], expected, atol=1e-9), (
        f"row 0 is {np.round(out[0], 4).tolist()} but tanh(b1 + three embedding rows) @ W2 + b2 "
        f"is {np.round(expected, 4).tolist()} — check the tanh and that all three lookups are added"
    )
    zeros = {name: np.zeros_like(value) for name, value in p.items()}
    zeros["b2"] = np.arange(VOCAB, dtype=float)
    assert np.allclose(policy_logits(zeros, rem, prev, pos), zeros["b2"][None, :]), (
        "with every weight zero except b2, each row of logits must equal b2 — is b2 added once per row?"
    )
    q = copy_params(p)
    q["b1"] = np.linspace(-1.0, 1.0, HIDDEN)
    q["b2"] = np.linspace(-2.0, 2.0, VOCAB)
    hidden = np.tanh(q["b1"] + q["rem_emb"][rem[1]] + q["prev_emb"][prev[1]] + q["pos_emb"][pos[1]])
    assert np.allclose(policy_logits(q, rem, prev, pos)[1], hidden @ q["W2"] + q["b2"], atol=1e-9), (
        "with non-zero b1 and b2 row 1 is wrong: b1 is added INSIDE the tanh, b2 after the output layer"
    )
    assert np.allclose(np.exp(log_softmax(out)).sum(axis=1), 1.0), "log_softmax of your logits must normalise"


_try("exercise 1", _check_policy_logits)

## 4. Exercise 2 — `logprob_grad()`, the backward pass

Policy-gradient methods never differentiate a reward. They differentiate `log π(token | context)`, weighted by
a number the algorithm chooses. Every objective in this lesson — the surrogate, the KL term, the length
normalisation — ends up as one **weight per token row**, and the whole update is the gradient of

`J(θ) = Σ_n  w_n · log π_θ(action_n | context_n)`

with respect to the parameters, for a weight vector `w` that does not depend on `θ` (the algorithm treats the
weights as constants: that is what "detach" means in a deep-learning framework). Derive it by hand:

- `∂ log π(a) / ∂ logits = onehot(a) − softmax(logits)`, times `w_n`;
- the output layer gives `∂J/∂b2`, `∂J/∂W2 = hᵀ · dlogits`, and the gradient flowing back into `h`;
- `tanh' = 1 − h²` turns that into `dpre`, and `dpre` is the gradient of `b1` and of every embedding row
  that was looked up. An embedding row looked up by several rows of the batch receives the SUM of their
  gradients.

The check below proves your backward pass against central finite differences of your own forward pass, block by
block, so a wrong block is named.

<details><summary>💡 Hint 1 — what to think about</summary>

Recompute the forward pass inside the function (you need `h` and the probabilities), then work backwards one
stage at a time. Keep a running list of which array each gradient has the shape of: the gradient of every
parameter has exactly that parameter's shape.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

`dlogits` has shape `(n, VOCAB)`: minus the probabilities, scaled by the weight of each row, with the weight
added back at the column of the emitted action. `W2` and `b2` come from `dlogits` and `h`; the gradient into
`h` comes from `dlogits` and `W2`. Multiply by `1 − h²` to get `dpre`. For the three embeddings, a row of
`dpre` belongs to whichever table row that decision looked up — several decisions can share one, so
accumulate with `np.add.at` rather than plain assignment.
</details>

In [ ]:
def logprob_grad(params: dict[str, np.ndarray], rem: np.ndarray, prev: np.ndarray,
                 pos: np.ndarray, action: np.ndarray, weights: np.ndarray) -> dict[str, np.ndarray]:
    """Gradient of `sum_n weights[n] * log pi(action[n] | rem[n], prev[n], pos[n])`, by hand.

    `weights` is treated as a constant (nothing is differentiated through it). Returns a dict with
    the same keys and shapes as `params`.

    Returns: {"rem_emb": ..., "prev_emb": ..., "pos_emb": ..., "b1": ..., "W2": ..., "b2": ...}.

    Example:
        >>> p = init_params(0)
        >>> g = logprob_grad(p, np.array([3]), np.array([BOS]), np.array([0]), np.array([4]), np.array([1.0]))
        >>> g["b2"].shape, round(float(g["b2"].sum()), 6)
        ((11,), 0.0)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def token_logprobs(params: dict[str, np.ndarray], rows: Rows) -> np.ndarray:
    """log pi(action | context) for every row. Given; it uses your policy_logits."""
    logp = log_softmax(policy_logits(params, rows.rem, rows.prev, rows.pos))
    return logp[np.arange(len(rows.action)), rows.action]


def finite_difference_gradient(f: Callable[[dict], float], params: dict[str, np.ndarray],
                               h: float = 1e-6) -> dict[str, np.ndarray]:
    """Central-difference gradient of a scalar function of the parameters. Given. Slow and exact enough."""
    grads = {}
    for name, array in params.items():
        g = np.zeros_like(array)
        for idx in np.ndindex(*array.shape):
            old = array[idx]
            array[idx] = old + h
            up = f(params)
            array[idx] = old - h
            down = f(params)
            array[idx] = old
            g[idx] = (up - down) / (2 * h)
        grads[name] = g
    return grads


_GRAD_ADVICE = {
    "b2": "dlogits = w * (onehot(action) - softmax): it is minus the probabilities, plus the weight at the action",
    "W2": "the gradient of W2 is hidden.T @ dlogits, summed over the rows",
    "b1": "dpre = (dlogits @ W2.T) * (1 - hidden**2); b1 is dpre summed over the rows",
    "rem_emb": "each embedding row gets the SUM of dpre over every decision that looked it up: use np.add.at",
    "prev_emb": "each embedding row gets the SUM of dpre over every decision that looked it up: use np.add.at",
    "pos_emb": "each embedding row gets the SUM of dpre over every decision that looked it up: use np.add.at",
}


def _gradient_report(params, rows: Rows, weights, tol: float = 1e-6) -> str | None:
    """None if `logprob_grad` matches finite differences on these rows, else a message naming the block."""
    analytic = logprob_grad(params, rows.rem, rows.prev, rows.pos, rows.action, weights)
    numeric = finite_difference_gradient(lambda q: float((weights * token_logprobs(q, rows)).sum()),
                                         copy_params(params))
    for name in PARAM_NAMES:
        assert name in analytic and analytic[name].shape == params[name].shape, (
            f"logprob_grad must return a '{name}' gradient of shape {params[name].shape}"
        )
        err = float(np.abs(analytic[name] - numeric[name]).max())
        if err > tol:
            return (f"the '{name}' gradient is off by up to {err:.2e} against finite differences — "
                    + _GRAD_ADVICE[name])
    return None


def _check_logprob_grad() -> None:
    params = init_params(3)
    rows = Rows(rem=np.array([3, 3, 0, 9, 1]), prev=np.array([BOS, 4, 4, 2, 10]),
                pos=np.array([0, 1, 1, 2, 3]), action=np.array([4, 10, 7, 0, 3]), seq=np.zeros(5, int))
    weights = np.array([1.0, -0.5, 2.0, 0.25, -1.5])
    problem = _gradient_report(params, rows, weights)
    assert problem is None, problem
    zero = logprob_grad(params, rows.rem, rows.prev, rows.pos, rows.action, np.zeros(5))
    assert all(np.abs(g).max() == 0 for g in zero.values()), "all-zero weights must give an all-zero gradient"
    print(f"exercise 2 looks right: every block of the gradient matches finite differences "
          f"(tolerance 1e-6) across {n_parameters(params)} parameters")


_try("exercise 2", _check_logprob_grad, needs=("exercise 1",))

## 5. Exercise 3 — `sample_group()`, group sampling

GRPO's first move is to sample `G` completions for each prompt from the CURRENT policy, one token at a time.
`sample_group(params, targets, G, rng)` returns an integer array of shape `(len(targets) * G, LMAX)`: the
completions for `targets[0]` first (`G` of them), then `targets[1]`, and so on. A completion is its tokens up
to and including `EOS`, then `-1` padding; one that never emits `EOS` fills all `LMAX` columns.

At every position, every still-running completion needs its context: the remaining sum (the target minus its
digits so far; `EOS` subtracts nothing), the previous token (`BOS` at the start), and the position. Sample the
next token from the softmax by inverse-CDF sampling: draw one uniform number per completion, and take the
first token whose cumulative probability reaches it. To keep every run reproducible, draw `rng.random(n)` for
all `n` completions once per position, in position order, whether or not a completion is still running.

<details><summary>💡 Hint 1 — what to think about</summary>

Loop over positions, not over completions: at each position, work on all `n` completions at once with
arrays. Three arrays carry the state between positions: how much of the target remains, the previous token,
and which completions are still running. A finished completion must not consume a token, and its column stays
at `-1`.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Get the logits for all rows from `policy_logits`, turn them into probabilities, take a running total along the
vocabulary axis, and count in how many columns that running total is still below the uniform draw: that count
is the sampled token (clip it to the last token in case rounding leaves the total a hair under one). Write
`-1` for rows that had already finished. Then update the state: subtract the sampled digit (not EOS) from
the remaining sum, remember the token as the previous one, and mark the row finished if it was EOS.
</details>

In [ ]:
def sample_group(params: dict[str, np.ndarray], targets: np.ndarray, group_size: int,
                 rng: np.random.Generator) -> np.ndarray:
    """Sample `group_size` completions for every target from the current policy.

    Returns an int array of shape (len(targets) * group_size, LMAX). Row `i` belongs to
    `targets[i // group_size]`. A completion lists its tokens through EOS, then -1 padding; one
    that never emits EOS has no -1 and no EOS. Draw `rng.random(n)` once per position, in order.

    Example:
        >>> toks = sample_group(init_params(0), np.array([4, 6]), 3, np.random.default_rng(0))
        >>> toks.shape
        (6, 8)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_sample_group() -> None:
    def sample_group(params, targets, group_size, rng):       # yours, inside a budget of LMAX decisions per completion
        n = len(targets) * group_size
        with _budget("policy_logits", LMAX * n, rows=True, mistake=(
                f"sample_group asked the policy for more than {LMAX} decisions per completion: does every completion "
                f"stop after at most LMAX = {LMAX} tokens, EOS or not? A loop that runs until every completion has "
                "emitted EOS never ends for a policy that never emits it")):
            return globals()["sample_group"](params, targets, group_size, rng)

    p = init_params(0)
    toks = sample_group(p, np.array([4, 6, 8]), 5, np.random.default_rng(1))
    assert toks.shape == (15, LMAX), f"expected shape (15, {LMAX}), got {toks.shape}"
    assert toks.min() >= -1 and toks.max() <= EOS, "tokens must be -1 (padding) or 0-10"
    for i, row in enumerate(toks):
        ends = np.nonzero(row == EOS)[0]
        if len(ends):
            assert (row[ends[0] + 1:] == -1).all() and (row[:ends[0] + 1] >= 0).all(), (
                f"completion {i} is {row.tolist()}: after the first EOS every entry must be -1, "
                "and before it none may be"
            )
        else:
            assert (row >= 0).all(), f"completion {i} never ended but has -1 padding: {row.tolist()}"
    always_three = copy_params(p)
    for name in ("rem_emb", "prev_emb", "pos_emb", "W2"):
        always_three[name][:] = 0.0
    always_three["b2"][:] = -30.0
    always_three["b2"][3] = 30.0
    assert (sample_group(always_three, np.array([5]), 4, np.random.default_rng(0)) == 3).all(), (
        f"a policy that almost surely emits token 3 must give all 3s and never EOS, so {LMAX} tokens each"
    )
    always_eos = copy_params(always_three)
    always_eos["b2"][:] = -30.0
    always_eos["b2"][EOS] = 30.0
    stops = sample_group(always_eos, np.array([5]), 4, np.random.default_rng(0))
    assert (stops[:, 0] == EOS).all() and (stops[:, 1:] == -1).all(), (
        "a policy that almost surely emits EOS first must give [EOS, -1, -1, ...] for every completion"
    )
    stop_when_overshot = copy_params(always_three)     # EOS only in the "overshot" remaining-sum class
    stop_when_overshot["b2"][:] = 0.0
    stop_when_overshot["b2"][EOS] = -30.0
    stop_when_overshot["rem_emb"][0, 0] = 40.0
    stop_when_overshot["W2"][0, EOS] = 60.0
    stop_when_overshot["b2"][DIGITS - 1] = -30.0        # never the digit 9, so every run has digits to sum
    target = 7
    out = sample_group(stop_when_overshot, np.array([target]), 40, np.random.default_rng(2))
    for row in out:
        digits = row[:np.nonzero(row == EOS)[0][0]] if (row == EOS).any() else row
        assert (row == EOS).any() and digits.sum() > target and digits[:-1].sum() <= target, (
            f"with EOS only possible after overshooting the target {target}, a completion must stop right "
            f"after its digits first exceed it; got {row.tolist()} — is the remaining sum updated with each "
            "digit, and not with EOS?"
        )
    by_position = {k: np.zeros_like(v) for k, v in p.items()}
    by_position["b2"][:] = -40.0
    schedule = [5, 1, 4, 9, 0, 2, 7, 3]
    for t, token in enumerate(schedule):
        by_position["pos_emb"][t, t] = 50.0
        by_position["W2"][t, token] = 80.0
    forced = sample_group(by_position, np.array([2, 8]), 2, np.random.default_rng(0))
    assert (forced == np.array(schedule)).all(), (
        f"a policy that emits {schedule} by position must give exactly that; got {forced[0].tolist()} — "
        "is the loop index t passed as the position?"
    )
    chain = {k: np.zeros_like(v) for k, v in p.items()}
    chain["b2"][:] = -40.0
    for digit in range(DIGITS):                  # after digit d, emit (d + 1) mod 10; from BOS, emit 0
        chain["prev_emb"][digit, digit] = 50.0
        chain["W2"][digit, (digit + 1) % DIGITS] = 80.0
    chain["prev_emb"][BOS, 10] = 50.0
    chain["W2"][10, 0] = 80.0
    counting = sample_group(chain, np.array([4]), 2, np.random.default_rng(0))
    assert (counting == np.arange(LMAX)).all(), (
        f"a policy that repeats 'previous token plus one' must count 0, 1, 2, ...; got {counting[0].tolist()} — "
        "is the token just sampled fed back as the previous token (BOS before the first)?"
    )
    ordered = sample_group(stop_when_overshot, np.array([8, 2]), 15, np.random.default_rng(5))
    for i, row in enumerate(ordered):
        goal = [8, 2][i // 15]
        digits = row[:np.nonzero(row == EOS)[0][0]] if (row == EOS).any() else row
        assert (row == EOS).any() and digits.sum() > goal and digits[:-1].sum() <= goal, (
            f"completion {i} belongs to target {goal}: the first 15 rows are for targets[0], the next 15 for targets[1]"
        )
    draws = sample_group(p, np.array([5]), 6000, np.random.default_rng(3))[:, 0]
    freq = np.bincount(draws, minlength=VOCAB) / len(draws)
    first = softmax(policy_logits(p, np.array([rem_class(5)]), np.array([BOS]), np.array([0])))[0]
    assert np.abs(freq - first).max() < 0.03, (
        f"the first token's empirical frequencies {np.round(freq, 3).tolist()} do not match the policy's "
        f"probabilities {np.round(first, 3).tolist()}: sample from the softmax, with inverse-CDF sampling"
    )
    a = sample_group(p, np.array([4, 6]), 4, np.random.default_rng(9))
    b = sample_group(p, np.array([4, 6]), 4, np.random.default_rng(9))
    assert (a == b).all(), "the same rng seed must give the same completions: draw only from the rng you were given"


_try("exercise 3", _check_sample_group, needs=("exercise 1",))

## 6. Exercise 4 — the verifier, `check_completion()`

The reward comes from a program, not from a model. `check_completion(target, tokens, verifier)` looks at one
completion (a row of the array `sample_group` returns) and answers two questions: did it **terminate** (does an
`EOS` appear?) and is it **correct**?

- the digits are the tokens before the first `EOS`; `-1` padding and everything after `EOS` is ignored;
- a completion that never emits `EOS` is not terminated, and is never correct, whatever its digits sum to;
- with `verifier="strict"` it is correct when the digits sum to exactly `target`;
- with `verifier="loophole"` it is correct when the digits sum to AT MOST `target`. This second verifier is a
  deliberate bug, the kind a hurried author writes when they mean "the digits must not overshoot" and forget
  "and must reach it". Section 19 trains a policy against it and measures what happens.

The given `completion_reward` turns the two answers into a number: `FORMAT_REWARD` for terminating plus
`CORRECT_REWARD` for being correct, so rewards take the values 0, `FORMAT_REWARD` and their sum. That mirrors
the accuracy-plus-format shape of DeepSeek-R1-Zero's rule-based reward (`r1-rule-based-rewards`).

<details><summary>💡 Hint 1 — what to think about</summary>

Find where the first `EOS` is, if there is one. Everything before it is the answer; everything from it onwards
is not. What are `terminated` and `correct` when there is no `EOS` at all?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Split the row at the first `EOS`. Sum the digits before it and compare with the target: equality for the strict
verifier, "not more than" for the loophole one. An empty answer (the very first token is `EOS`) sums to zero.
Return both flags as plain Python booleans, `correct` only ever true when `terminated` is.
</details>

In [ ]:
def check_completion(target: int, tokens, verifier: str = "strict") -> tuple[bool, bool]:
    """Check one completion. Returns `(terminated, correct)`, both plain bools.

    `tokens` is a sequence of ints: digits, then EOS, then -1 padding (or no EOS at all). The
    verifier is "strict" (digits sum to exactly `target`) or "loophole" (digits sum to at most
    `target`). A completion that never emits EOS is neither terminated nor correct.

    Example:
        >>> check_completion(7, [3, 4, EOS, -1, -1, -1, -1, -1])
        (True, True)
        >>> check_completion(7, [3, 4, 2, 1, 5, 6, 7, 8])
        (False, False)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def completion_reward(target: int, tokens, verifier: str = "strict") -> float:
    """Format reward for terminating plus correctness reward for being right. Given."""
    terminated, correct = check_completion(target, tokens, verifier)
    return FORMAT_REWARD * terminated + CORRECT_REWARD * correct


def group_rewards(targets: np.ndarray, group_size: int, tokens: np.ndarray,
                  verifier: str = "strict") -> np.ndarray:
    """The reward of every completion, shape (len(tokens),). Given."""
    per_row = np.repeat(np.asarray(targets), group_size)
    return np.array([completion_reward(int(t), row.tolist(), verifier) for t, row in zip(per_row, tokens)])


def completion_lengths(tokens: np.ndarray) -> np.ndarray:
    """Tokens per completion, EOS included and padding not. Given."""
    return (tokens >= 0).sum(axis=1)


def _check_check_completion() -> None:
    pad = [-1] * 8

    def row(*toks):
        return list(toks) + pad[:LMAX - len(toks)]

    assert check_completion(7, row(3, 4, EOS)) == (True, True), "3 + 4 = 7 then EOS is terminated and correct"
    assert check_completion(7, row(3, 5, EOS)) == (True, False), "3 + 5 is not 7: terminated but not correct"
    assert check_completion(7, row(3, 4, 2, 1, 5, 6, 7, 1)) == (False, False), (
        "a completion with no EOS is not terminated, and never correct"
    )
    assert check_completion(7, row(7, 0, 0, EOS)) == (True, True), "zeros add nothing: 7 + 0 + 0 = 7"
    assert check_completion(7, row(EOS)) == (True, False), (
        "an empty answer sums to 0, which is not 7: with the strict verifier it is terminated but wrong"
    )
    assert check_completion(7, row(2, EOS, 5, 5)) == (True, False), "digits after the first EOS do not count"
    assert check_completion(9, row(4, 5, EOS)) == (True, True) and check_completion(10, row(4, 5, EOS)) == (True, False), (
        "the sum is compared with the target exactly; do not add the EOS token's own value 10 to the sum"
    )
    assert check_completion(7, row(EOS), verifier="loophole") == (True, True), (
        "the loophole verifier accepts any sum that does not exceed the target, so the empty answer passes"
    )
    assert check_completion(7, row(4, 4, EOS), verifier="loophole") == (True, False), "8 exceeds 7: not correct"
    assert check_completion(7, row(3, 4, 2, 1, 5, 6, 7, 1), verifier="loophole") == (False, False), (
        "even the loophole verifier needs EOS: no EOS, not terminated, not correct"
    )
    assert check_completion(7, row(3, 4, EOS), verifier="loophole") == (True, True), (
        "the loophole verifier also accepts a sum that EQUALS the target: it is 'at most'"
    )
    assert check_completion(7, [1, 1, 1, 1, 1, 1, 1, 0]) == (False, False), (
        "eight tokens summing to 7 with no EOS: a completion that never terminates is never correct, even when its digits add up"
    )
    assert check_completion(5, row(2, 3, EOS, 4, EOS)) == (True, True), "only the FIRST EOS ends the answer"
    result = check_completion(7, row(3, 4, EOS))
    assert all(type(flag) is bool for flag in result), "return plain Python bools, not numpy booleans"


_try("exercise 4", _check_check_completion)

## 7. Exercise 5 — group-relative advantages, and the zero-variance group

The advantage of completion `i` in a group is its reward minus the group's mean reward, divided by the
group's standard deviation plus a small epsilon (`grpo-advantage-normalisation`). Use the population standard
deviation (`np.std`'s default, `ddof=0`); the source does not say which estimator it means, so this lesson
fixes one and says so. `normalise_std=False` skips the division, leaving `r - mean`: one of the variants section 16 measures.

A group whose rewards are all equal has zero standard deviation and a zero numerator. The advantage must be
exactly `0.0`, not `nan` and not a huge number: the epsilon in the denominator is what makes that safe, and the
group teaches the policy nothing. (In floating point the mean of equal rewards can miss them by a rounding error,
three rewards of 0.1 for example; with this lesson's rewards and group sizes it does not.)
`zero_variance_fraction` counts how often that happens, because it is a measurable cost of the recipe
(`dapo-zero-advantage`).

`rewards` is a flat array of `n_groups * group_size` values, group after group, in the order `sample_group`
produced its completions.

<details><summary>💡 Hint 1 — what to think about</summary>

Reshape the flat rewards to `(n_groups, group_size)` so that a mean and a standard deviation along the last
axis are per-group numbers, with `keepdims` so they broadcast back. Then flatten again.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Centre each group on its own mean. If `normalise_std` is set, divide by that group's standard deviation plus
`eps` (add the epsilon to the standard deviation, not to the variance). For `zero_variance_fraction`, a group
has zero variance when every one of its rewards equals the first, which for the discrete rewards in this
lesson is an exact comparison; return the fraction of groups that do.
</details>

In [ ]:
def group_advantages(rewards: np.ndarray, group_size: int, eps: float = ADV_EPS,
                     normalise_std: bool = True) -> np.ndarray:
    """Group-relative advantage of every completion. `rewards` is flat, group after group.

    `A_i = (r_i - mean) / (std + eps)` with the population std (`ddof=0`); with
    `normalise_std=False`, `A_i = r_i - mean`. A group of identical rewards gets exactly 0.

    Returns: a flat array shaped like `rewards`.

    Example:
        >>> group_advantages(np.array([1.0, 0.0, 0.0, 0.0]), 4, normalise_std=False).tolist()
        [0.75, -0.25, -0.25, -0.25]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def zero_variance_fraction(rewards: np.ndarray, group_size: int) -> float:
    """The fraction of groups in which every completion has the same reward.

    Example:
        >>> zero_variance_fraction(np.array([1.0, 1.0, 0.0, 1.0]), 2)
        0.5
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_group_advantages() -> None:
    adv = group_advantages(np.array([1.0, 0.0, 0.0, 0.0]), 4, eps=0.0)
    mean, std = 0.25, np.sqrt(0.1875)
    assert np.allclose(adv, [(1 - mean) / std] + [(0 - mean) / std] * 3), (
        f"got {np.round(adv, 4).tolist()}; expected (r - mean) / std with the population std (ddof=0): "
        f"{np.round((1 - mean) / std, 4)} and then {np.round(-mean / std, 4)} three times"
    )
    flat = group_advantages(np.array([0.5, 0.5, 0.5, 0.5, 1.0, 0.0]), 2)
    assert np.isfinite(flat).all() and (flat[:4] == 0.0).all(), (
        f"a group of identical rewards must have advantage exactly 0.0, got {flat[:4].tolist()} — "
        "add the epsilon to the standard deviation in the denominator, so 0 / (0 + eps) = 0"
    )
    assert abs(flat[4] + flat[5]) < 1e-12 and flat[4] > 0 > flat[5], "a group's advantages sum to zero"
    assert np.allclose(group_advantages(np.array([3.0, 1.0]), 2, normalise_std=False), [1.0, -1.0]), (
        "with normalise_std=False the advantage is just reward minus the group mean"
    )
    loose = group_advantages(np.array([1.0, 0.0]), 2, eps=0.5)
    assert abs(loose[0] - 0.5 / (0.5 + 0.5)) < 1e-12, "the epsilon is added to the standard deviation"
    two = group_advantages(np.array([1.0, 0.0, 3.0, 2.0]), 2, eps=0.0)
    assert np.allclose(two, [1.0, -1.0, 1.0, -1.0]), (
        f"got {np.round(two, 3).tolist()}: each group is centred on its OWN mean and divided by its OWN std"
    )
    assert zero_variance_fraction(np.array([1.0, 1.0, 0.0, 1.0, 0.0, 0.0]), 2) == 2 / 3, (
        "count the groups whose rewards are all equal: here groups 0 and 2 of 3"
    )
    assert zero_variance_fraction(np.array([1.0, 0.0, 0.0, 1.0]), 2) == 0.0, "no group here has equal rewards"
    assert zero_variance_fraction(np.array([0.1, 0.1, 1.1, 0.1, 0.0, 0.1]), 3) == 0.0, (
        "compare EVERY reward in a group, not just the first two, and compare exactly: 0.0 and 0.1 are different rewards"
    )
    assert zero_variance_fraction(np.array([0.1, 0.1, 0.1, 0.5, 0.55, 0.5]), 3) == 0.5, (
        "three rewards of 0.1 are a tied group (compare the rewards themselves: a computed std of equal floats need not "
        "come out exactly 0), and 0.5 and 0.55 are different rewards (no tolerance)"
    )


_try("exercise 5", _check_group_advantages)

## 8. Exercise 6 — the clipped surrogate, `clipped_surrogate()`

GRPO takes several gradient steps on one batch of samples. After the first step the policy has moved, so the
samples are slightly off-policy, and the objective corrects for it with the ratio `ρ = π_θ(token) / π_old(token)`
and clips it (the PPO trick GRPO inherits, `ppo-clip-stabilises`). For one token with advantage `A`, the objective is

`min( ρ·A ,  clip(ρ, 1−ε, 1+ε)·A )`

The clip is a trust region that only bites in one direction: it stops rewarding a token once `ρ` has already
moved far in the direction `A` wanted, and does nothing when `ρ` has moved the wrong way. In terms of the
gradient with respect to `log π_θ`: it equals `ρ·A` on the tokens where the `min` picks the unclipped term
(including exactly at the boundary, where the two terms are equal), and `0` where it picks the clipped,
constant, term.

You return the objective per token, the gradient weight per token, and which tokens are still *active*. The
weight is what `logprob_grad` will take. (Inside the update, `ρ` is `exp(logp_new − logp_old)`, where
`logp_old` is a constant.)

<details><summary>💡 Hint 1 — what to think about</summary>

Work out the four cases on paper: `A` positive or negative, `ρ` above `1+ε`, below `1−ε`, or inside. Which of
the two terms is smaller in each? Where the smaller term is the clipped one, the objective is flat in `ρ`, so
its gradient is zero.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Compute both terms as arrays, `unclipped = ρ·A` and `clipped = clip(ρ, ...)·A`, and the objective is their
element-wise minimum. A token is active when the unclipped term is not larger than the clipped one, so that
the boundary counts as active. The gradient weight is `ρ·A` where active and `0` elsewhere: it is
`∂(ρ·A)/∂ log π = ρ·A`, because `∂ρ/∂ log π = ρ`.
</details>

In [ ]:
def clipped_surrogate(logp_new: np.ndarray, logp_old: np.ndarray, adv: np.ndarray,
                      clip_eps: float) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """PPO/GRPO clipped surrogate for a batch of tokens.

    `logp_new`, `logp_old`, `adv` are arrays of one entry per token. Returns
    `(objective, grad_weight, active)`: the per-token objective; the derivative of the objective with
    respect to `logp_new` (`rho * adv` where the unclipped term is the smaller or equal one, else 0);
    and a boolean array, True where that derivative is not clipped away. When `adv` is exactly 0 the
    two terms are always equal, so the token is active.

    Example:
        >>> obj, w, active = clipped_surrogate(np.array([0.0]), np.array([0.0]), np.array([2.0]), 0.2)
        >>> float(obj[0]), float(w[0]), bool(active[0])
        (2.0, 2.0, True)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_clipped_surrogate() -> None:
    eps = 0.2
    log = np.log
    cases = [
        # (rho, adv, active, expected objective)
        (1.5, 2.0, False, 1.2 * 2.0),       # A > 0, rho above 1 + eps: clipped, flat
        (0.5, 2.0, True, 0.5 * 2.0),        # A > 0, rho below 1 - eps: the unclipped term is the smaller one
        (1.5, -2.0, True, 1.5 * -2.0),      # A < 0, rho above 1 + eps: the unclipped term is the smaller one
        (0.5, -2.0, False, 0.8 * -2.0),     # A < 0, rho below 1 - eps: clipped, flat
        (1.1, 2.0, True, 1.1 * 2.0),        # inside the band: plain ratio times advantage
        (1.1, -2.0, True, 1.1 * -2.0),
    ]
    for rho, adv, want_active, want_obj in cases:
        obj, weight, active = clipped_surrogate(log(np.array([rho])), np.zeros(1), np.array([adv]), eps)
        side = "above" if rho > 1 else "below"
        assert abs(obj[0] - want_obj) < 1e-12, (
            f"rho={rho} ({side} 1), A={adv}: objective {obj[0]:.6f}, expected {want_obj:.6f} — "
            "take the MINIMUM of the unclipped and clipped terms"
        )
        assert bool(active[0]) == want_active, (
            f"rho={rho}, A={adv}: active should be {want_active}; the clip only bites when the ratio has "
            "already moved past the band in the direction the advantage wanted"
        )
        expect_w = rho * adv if want_active else 0.0
        assert abs(weight[0] - expect_w) < 1e-12, f"rho={rho}, A={adv}: gradient weight {weight[0]:.4f}, expected {expect_w:.4f}"
    edge = clipped_surrogate(np.zeros(2), np.zeros(2), np.array([1.0, -1.0]), 0.0)
    assert edge[2].all(), "exactly at the boundary (eps = 0, rho = 1) the two terms are equal: count the token as active"
    zero_adv = clipped_surrogate(np.log(np.array([1.5])), np.zeros(1), np.zeros(1), eps)
    assert bool(zero_adv[2][0]), "with an advantage of exactly 0 the two terms are equal: the token is active"
    ratios = np.array([0.7, 1.05, 1.4])
    adv = np.array([1.5, -0.5, 1.0])
    lp = np.log(ratios)
    obj, weight, _ = clipped_surrogate(lp, np.zeros(3), adv, eps)
    step = 1e-6
    numeric = (clipped_surrogate(lp + step, np.zeros(3), adv, eps)[0]
               - clipped_surrogate(lp - step, np.zeros(3), adv, eps)[0]) / (2 * step)
    assert np.allclose(weight, numeric, atol=1e-6), (
        f"grad_weight {np.round(weight, 4).tolist()} disagrees with a finite difference of your own "
        f"objective {np.round(numeric, 4).tolist()}: the weight is rho * A, not A"
    )


_try("exercise 6", _check_clipped_surrogate)

## 9. Exercise 7 — the KL penalty and the k3 estimator, `k3_kl()`

GRPO does not put the KL penalty in the reward. It subtracts `β · KL(π_θ ‖ π_ref)` from the objective,
estimated on the sampled tokens (`grpo-kl-estimator`, `grpo-kl-unbiased-estimator`). With `u = log π_ref(token) − log π_θ(token)`, the
estimator the source uses, named k3 in Schulman's note (`k3-estimator`), is

`k3 = exp(u) − u − 1`.

It is never negative (`k3-nonnegative`), and its expectation over samples drawn from `π_θ` is the KL itself. You return the
estimator per token and its derivative with respect to `log π_θ(token)`; the update needs the derivative,
because the penalty enters the gradient as a per-token weight `−β · dk3/dlogp`. Differentiate by hand:
`u` depends on `log π_θ` with slope `−1`.

<details><summary>💡 Hint 1 — what to think about</summary>

Write `k3` as a function of `u`, differentiate with respect to `u`, then multiply by `du/dlogp`. Check
the sign of the result at `u = 0` (policy equal to reference): the penalty should have no pull there.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Differentiate with respect to `u` (the exponential is its own derivative), then apply the chain rule through
`u = log π_ref − log π_θ`, whose slope in `log π_θ` is negative. Check the sign of your answer against the
story: when the policy has made a token MORE likely than the reference did, the penalty must slope
up with `log π_θ` (so that the update, which subtracts it, pulls the policy back), and it must vanish at `u = 0`.
</details>

In [ ]:
def k3_kl(logp: np.ndarray, logp_ref: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """The k3 estimator of KL(pi_theta || pi_ref) per token, and its derivative w.r.t. `logp`.

    `logp` and `logp_ref` are log-probabilities of the same sampled tokens under the policy and
    the reference. Returns `(k3, dk3_dlogp)`.

    Example:
        >>> k3, d = k3_kl(np.array([-1.0]), np.array([-1.0]))
        >>> float(k3[0]), float(d[0])
        (0.0, 0.0)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_k3_kl() -> None:
    logp = np.array([-2.0, -1.0, -0.3, -4.0])
    ref = np.array([-1.0, -1.0, -1.2, -3.0])
    k3, dk3 = k3_kl(logp, ref)
    u = ref - logp
    assert np.allclose(k3, np.exp(u) - u - 1), (
        f"k3 should be exp(u) - u - 1 with u = logp_ref - logp; got {np.round(k3, 4).tolist()}"
    )
    assert (k3 >= 0).all() and k3[1] == 0.0, "k3 is never negative, and is 0 when the policy equals the reference"
    step = 1e-6
    numeric = (k3_kl(logp + step, ref)[0] - k3_kl(logp - step, ref)[0]) / (2 * step)
    assert np.allclose(dk3, numeric, atol=1e-6), (
        f"dk3_dlogp {np.round(dk3, 4).tolist()} disagrees with the finite difference "
        f"{np.round(numeric, 4).tolist()}: u falls as logp rises, so the chain rule brings in a factor of -1 — check the sign"
    )
    pi = np.array([0.5, 0.3, 0.2])
    pref = np.array([0.2, 0.5, 0.3])
    per_token, _ = k3_kl(np.log(pi), np.log(pref))
    exact = float((pi * np.log(pi / pref)).sum())
    assert abs(float((pi * per_token).sum()) - exact) < 1e-12, (
        "averaged over tokens drawn from the policy, k3 must equal the exact KL(pi || pi_ref); "
        f"here {(pi * per_token).sum():.6f} against {exact:.6f}"
    )


_try("exercise 7", _check_k3_kl)

## 10. Exercise 8 — length normalisation, `length_scale()`

Each completion's token terms are added and then scaled by a per-completion factor `f_i`, and the whole loss is
the average over completions of `f_i · Σ_t (token term)`. The choice of `f_i` is a modelling decision with
consequences:

- `"seq"`, GRPO as published: `f_i = 1 / |o_i|`, the completion's own length. Each completion contributes an
  average per-token term, so a short and a long completion weigh the same.
- `"const"`, the variant Dr. GRPO proposes: `f_i = 1 / max_len`, a constant. Every token has the same weight
  whatever completion it sits in (`drgrpo-length-bias`, `drgrpo-constant-normaliser`).
- `"batch"`, token-level averaging as in DAPO: `f_i = 1 / mean(lengths)` over the whole batch, so that the
  loss is a per-token mean across the batch (`dapo-sample-level-loss`, `dapo-token-level-loss`).

Implement `length_scale(lengths, mode, max_len)`, returning one factor per completion. An unknown mode must
raise `ValueError`, not silently pick one.

<details><summary>💡 Hint 1 — what to think about</summary>

Three modes, three one-line formulas, and one thing that changes shape: two of them give a different number to
each completion, one gives every completion the same number. What array shape should each return?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Convert `lengths` to floats. `seq` is the reciprocal of each length, elementwise. `const` is a full-size
array holding the reciprocal of `max_len`. `batch` is a full-size array holding the reciprocal of the mean of
all the lengths. Anything else raises.
</details>

In [ ]:
def length_scale(lengths: np.ndarray, mode: str, max_len: int = LMAX) -> np.ndarray:
    """One factor per completion for the loss's length normalisation.

    "seq": 1 / length of that completion; "const": 1 / max_len for all; "batch": 1 / mean length of
    the batch for all. Raises ValueError for any other mode.

    Example:
        >>> length_scale(np.array([2, 4]), "seq").tolist()
        [0.5, 0.25]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_length_scale() -> None:
    lengths = np.array([2, 4, 6, 2])
    seq = length_scale(lengths, "seq", 8)
    assert np.allclose(seq, [0.5, 0.25, 1 / 6, 0.5]), f"'seq' is 1 / each completion's own length; got {seq.tolist()}"
    const = length_scale(lengths, "const", 8)
    assert const.shape == (4,) and np.allclose(const, 1 / 8), (
        f"'const' is 1 / max_len (the argument, here 8, not the longest completion) for every completion; got {const.tolist()}"
    )
    batch = length_scale(lengths, "batch", 8)
    assert batch.shape == (4,) and np.allclose(batch, 1 / 3.5), (
        f"'batch' is 1 / the mean length of the whole batch (here 3.5) for every completion; got {batch.tolist()}"
    )
    try:
        length_scale(lengths, "nonsense", 8)
    except ValueError:
        pass
    else:
        raise AssertionError("an unknown mode must raise ValueError")


_try("exercise 8", _check_length_scale)

## 11. Putting it together: the rollout, the optimiser, and the update

The next two exercises assemble the pieces. First, the given parts. A **rollout** bundles everything one
training step needs about a batch of completions: the token rows, each completion's advantage and length, and
the log-probabilities of every token under the policy that sampled it (`logp_old`, a constant from now on) and
under the frozen reference (`logp_ref`). An **optimiser** turns a gradient into a step; the lesson uses Adam,
and a plain gradient-ascent `SGD` for the checks, because its step is exactly `lr` times the gradient. Both
ASCEND, since GRPO maximises its objective. A **configuration** holds the hyperparameters.

A training step is then: sample the groups, score them, compute advantages, build the rollout, and run
`epochs` gradient steps on it. Each epoch recomputes the current log-probabilities, gets each token's weight
from the clipped surrogate and the KL penalty, scales by the length normalisation, and takes a gradient step.

In [ ]:
class GRPOConfig(NamedTuple):
    steps: int = 100               # training steps (each: sample, score, update)
    group_size: int = 8            # G: completions per prompt
    epochs: int = 2                # gradient steps per batch
    clip_eps: float = 0.2          # the clip band around a ratio of 1
    beta: float = 0.02             # the KL penalty coefficient
    lr: float = 0.01               # Adam step size
    length_norm: str = "seq"       # "seq" (GRPO), "const" (Dr. GRPO) or "batch" (token-level)
    normalise_std: bool = True     # divide advantages by the group's standard deviation?
    verifier: str = "strict"       # "strict", or "loophole" (section 14)


class Rollout(NamedTuple):
    rows: Rows
    advantages: np.ndarray   # (n_completions,) one per completion
    lengths: np.ndarray      # (n_completions,) tokens per completion, EOS included
    logp_old: np.ndarray     # (n_rows,) under the policy that sampled the batch
    logp_ref: np.ndarray     # (n_rows,) under the frozen reference policy


class SGD:
    """Plain gradient ASCENT: params += lr * grad. Used by the checks because its step is exact."""

    def __init__(self, lr: float):
        self.lr = lr

    def step(self, params: dict[str, np.ndarray], grads: dict[str, np.ndarray]) -> None:
        for name in params:
            params[name] += self.lr * grads[name]


class Adam:
    """Adam, ascending. Updates `params` in place."""

    def __init__(self, params: dict[str, np.ndarray], lr: float, beta1: float = 0.9,
                 beta2: float = 0.999, eps: float = 1e-8):
        self.lr, self.beta1, self.beta2, self.eps, self.t = lr, beta1, beta2, eps, 0
        self.m = {name: np.zeros_like(v) for name, v in params.items()}
        self.v = {name: np.zeros_like(v) for name, v in params.items()}

    def step(self, params: dict[str, np.ndarray], grads: dict[str, np.ndarray]) -> None:
        self.t += 1
        for name in params:
            self.m[name] = self.beta1 * self.m[name] + (1 - self.beta1) * grads[name]
            self.v[name] = self.beta2 * self.v[name] + (1 - self.beta2) * grads[name] ** 2
            m_hat = self.m[name] / (1 - self.beta1 ** self.t)
            v_hat = self.v[name] / (1 - self.beta2 ** self.t)
            params[name] += self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


def flatten_completions(targets: np.ndarray, group_size: int, tokens: np.ndarray) -> Rows:
    """Turn a (n, LMAX) sample array into one Rows entry per emitted token. Given."""
    n = tokens.shape[0]
    rem = np.repeat(np.asarray(targets), group_size).astype(int)
    prev = np.full(n, BOS)
    parts: dict[str, list[np.ndarray]] = {"rem": [], "prev": [], "pos": [], "action": [], "seq": []}
    for t in range(tokens.shape[1]):
        action = tokens[:, t]
        live = np.nonzero(action >= 0)[0]
        parts["rem"].append(rem_class(rem[live]))
        parts["prev"].append(prev[live])
        parts["pos"].append(np.full(len(live), t))
        parts["action"].append(action[live])
        parts["seq"].append(live)
        is_digit = (action >= 0) & (action < EOS)
        rem = np.where(is_digit, rem - action, rem)
        prev = np.where(action >= 0, action, prev)
    return Rows(*(np.concatenate(parts[key]).astype(int) for key in ("rem", "prev", "pos", "action", "seq")))


def make_rollout(params, ref_params, targets, group_size, tokens, advantages) -> Rollout:
    """Bundle a scored batch for the update. Given; it uses your policy_logits."""
    rows = flatten_completions(targets, group_size, tokens)
    return Rollout(rows, advantages, completion_lengths(tokens), token_logprobs(params, rows),
                   token_logprobs(ref_params, rows))


def step_rng(seed: int, step: int) -> np.random.Generator:
    """The random stream of one training step of one seed. Given: every run is reproducible from it."""
    return np.random.default_rng([SEED, seed, 1, step])

## 12. Exercise 9 — `grpo_update()`

Given a rollout, run `cfg.epochs` gradient steps. In every epoch:

1. `logp = token_logprobs(params, rollout.rows)`, the CURRENT policy's log-probabilities, recomputed each epoch;
2. the per-token advantage is the completion's advantage, looked up through `rows.seq`;
3. `clipped_surrogate(logp, logp_old, advantage, clip_eps)` gives the gradient weight `gw` per token;
4. `k3_kl(logp, logp_ref)` gives `dk3` per token;
5. the weight of each token row is `f · (gw − β·dk3) / n`, where `f = length_scale(lengths, cfg.length_norm)`
   looked up through `rows.seq`, and `n` is the number of COMPLETIONS in the rollout (the loss averages over
   completions);
6. `logprob_grad` turns the weights into a gradient, and `opt.step(params, grad)` ascends it, in place.

Return a dict with `"kl"`, the mean k3 over all token rows at the START of the first epoch (that is, of the
policy that sampled the batch), and `"clip_fraction"`, the fraction of token rows that were NOT active in the
LAST epoch.

<details><summary>💡 Hint 1 — what to think about</summary>

The whole update is a loop of `epochs` iterations around code you have already written. Which of the six
quantities are constants across the epochs, and which must be recomputed inside the loop? The one thing
that must not be recomputed is `logp_old`.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Precompute what does not change: the per-row length factor and the per-row advantage. Inside the loop, get
the current log-probabilities, then the surrogate's gradient weight and the KL derivative, then combine them:
the surrogate weight minus `beta` times the KL derivative, times the length factor, divided by the number of
completions. Remember the KL and the clip fraction as you go, in the epochs the description above names.
</details>

In [ ]:
def grpo_update(params: dict[str, np.ndarray], opt, rollout: Rollout, cfg: GRPOConfig) -> dict[str, float]:
    """Run `cfg.epochs` GRPO gradient steps on one rollout, updating `params` in place.

    Uses `token_logprobs`, `clipped_surrogate`, `k3_kl`, `length_scale`, `logprob_grad` and
    `opt.step`. Returns `{"kl": ..., "clip_fraction": ...}` as described above.

    Example: with `SGD(lr)`, `beta=0`, `epochs=1` and every advantage 0, `params` does not change.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_grpo_update() -> None:
    def grpo_update(params, opt, rollout, cfg):              # yours, inside a budget of backward passes
        with _budget("logprob_grad", 3 * cfg.epochs + 3, mistake=(
                f"grpo_update called logprob_grad more than {3 * cfg.epochs + 3} times for cfg.epochs = {cfg.epochs}: "
                "does the epoch loop stop after cfg.epochs gradient steps?")):
            return globals()["grpo_update"](params, opt, rollout, cfg)

    params = init_params(5)
    ref = init_params(6)
    targets = np.array([4, 6])
    tokens = sample_group(params, targets, 3, np.random.default_rng(0))
    rewards = group_rewards(targets, 3, tokens)
    adv = np.array([1.0, -0.5, -0.5, 0.2, 0.2, -0.4])
    rollout = make_rollout(params, ref, targets, 3, tokens, adv)
    n = len(adv)
    rows = rollout.rows

    def objective(q, cfg):
        logp = token_logprobs(q, rows)
        rho = np.exp(logp - rollout.logp_old)
        u = rollout.logp_ref - logp
        per_token = adv[rows.seq] * rho - cfg.beta * (np.exp(u) - u - 1)
        f = length_scale(rollout.lengths, cfg.length_norm, LMAX)[rows.seq]
        return float((f * per_token).sum() / n)

    for norm in ("seq", "const"):
        cfg = GRPOConfig(epochs=1, beta=0.05, length_norm=norm, clip_eps=0.2)
        numeric = finite_difference_gradient(lambda q: objective(q, cfg), copy_params(params))
        lr = 0.01
        trial = copy_params(params)
        grpo_update(trial, SGD(lr), rollout, cfg)
        for name in PARAM_NAMES:
            moved = (trial[name] - params[name]) / lr
            assert np.allclose(moved, numeric[name], atol=1e-6), (
                f"with length_norm='{norm}' the step on '{name}' is not lr times the gradient of "
                f"(1/n) sum_i f_i sum_t [A_i * rho - beta * k3]: off by up to "
                f"{np.abs(moved - numeric[name]).max():.2e}. Check the weight "
                "f * (gw - beta * dk3) / n, and that n counts completions, not token rows"
            )
    still = copy_params(params)
    stats = grpo_update(still, SGD(0.01), rollout._replace(advantages=np.zeros(n)),
                        GRPOConfig(epochs=2, beta=0.0))
    assert all(np.array_equal(still[k], params[k]) for k in PARAM_NAMES), (
        "with every advantage zero and beta = 0 nothing pulls on the parameters: they must not move"
    )
    kl_expected = float(k3_kl(rollout.logp_old, rollout.logp_ref)[0].mean())
    assert abs(stats["kl"] - kl_expected) < 1e-12, (
        f"stats['kl'] is {stats['kl']:.6f}; it should be the mean k3 of the sampling policy against the "
        f"reference at the START of the first epoch, {kl_expected:.6f}"
    )
    moved_big = copy_params(params)
    stats = grpo_update(moved_big, SGD(3.0), rollout, GRPOConfig(epochs=3, beta=0.0, clip_eps=0.2))
    assert abs(stats["kl"] - kl_expected) < 1e-12, (
        f"after three large epochs stats['kl'] is {stats['kl']:.6f}: it is the KL of the policy that SAMPLED the batch, "
        f"measured at the start of the first epoch ({kl_expected:.6f}), not after the steps"
    )
    assert 0.0 < stats["clip_fraction"] <= 1.0, (
        f"after large steps some ratios leave the clip band, so clip_fraction should be positive (got "
        f"{stats['clip_fraction']}); measure it in the LAST epoch, as the fraction of rows that are not active"
    )
    tight, loose = copy_params(params), copy_params(params)
    grpo_update(tight, SGD(0.05), rollout, GRPOConfig(epochs=2, beta=0.0, clip_eps=1e-6))
    grpo_update(loose, SGD(0.05), rollout, GRPOConfig(epochs=2, beta=0.0, clip_eps=10.0))
    assert any(not np.allclose(tight[k], loose[k]) for k in PARAM_NAMES), (
        "a clip band of 1e-6 and one of 10 must give different second epochs: read the band from cfg.clip_eps"
    )
    fresh = copy_params(params)
    stats = grpo_update(fresh, SGD(0.01), rollout, GRPOConfig(epochs=1))
    assert stats["clip_fraction"] == 0.0, "in the first epoch the ratio is exactly 1 everywhere: nothing is clipped"

    def clipped_objective(q, cfg):
        logp = token_logprobs(q, rows)
        rho = np.exp(logp - rollout.logp_old)
        u = rollout.logp_ref - logp
        a = adv[rows.seq]
        per_token = np.minimum(rho * a, np.clip(rho, 1 - cfg.clip_eps, 1 + cfg.clip_eps) * a) - cfg.beta * (np.exp(u) - u - 1)
        return float((length_scale(rollout.lengths, cfg.length_norm, LMAX)[rows.seq] * per_token).sum() / n)

    two = GRPOConfig(epochs=2, beta=0.05, length_norm="seq", clip_eps=0.2)
    once, twice = copy_params(params), copy_params(params)
    grpo_update(once, SGD(0.01), rollout, two._replace(epochs=1))
    grpo_update(twice, SGD(0.01), rollout, two)
    numeric = finite_difference_gradient(lambda q: clipped_objective(q, two), copy_params(once))
    for name in PARAM_NAMES:
        moved = (twice[name] - once[name]) / 0.01
        assert np.allclose(moved, numeric[name], atol=1e-6), (
            f"the SECOND epoch's step on '{name}' is not lr times the gradient of the same objective at the parameters "
            f"the first epoch left (off by up to {np.abs(moved - numeric[name]).max():.2e}): every epoch recomputes the "
            "current log-probabilities, for the ratio AND for the KL term, and logp_old stays fixed"
        )
    print("exercise 9 looks right: one epoch is exactly lr times the gradient of the GRPO objective, "
          f"under both length normalisations, on {len(rows.action)} token rows")


_try("exercise 9", _check_grpo_update,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 6", "exercise 7", "exercise 8"))

## 13. Exercise 10 — the training loop, `train_grpo()`

The loop is short once the pieces exist. For each step `s` in `range(cfg.steps)`:

1. draw the step's random stream with `step_rng(seed, s)` and sample the groups for every target with it;
2. score them (`group_rewards`) and compute the group-relative advantages;
3. build the rollout against the frozen reference, and run `grpo_update`;
4. record `step_metrics(...)` (given), which needs the batch, the rewards and the update's stats.

The reference policy is a COPY of the initial parameters, taken before the first update and never touched
again. Return `(params, history)`, `history` being the list of the per-step metric dicts.

<details><summary>💡 Hint 1 — what to think about</summary>

Two objects live across steps: the parameters being trained, with their optimiser (`Adam`, which needs the
parameters and `cfg.lr`), and the frozen reference. What goes wrong if the reference shares its arrays with
the parameters, given that the optimiser updates the parameters in place?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Before the loop: initial parameters from `init_params(seed)`, a `copy_params` of them as the reference, an
`Adam`. In the loop, in the order above: the step's rng into `sample_group`, the rewards under
`cfg.verifier`, the advantages under `cfg.normalise_std`, the rollout, the update; then append the
step's metrics. Metrics are computed from the batch that was sampled, before or after the update — they
describe the samples, and `step_metrics` needs only what is already in hand.
</details>

In [ ]:
def step_metrics(targets, group_size, tokens, rewards, stats, verifier) -> dict[str, float]:
    """What one training step reports. Given; it uses your check_completion and zero_variance_fraction."""
    per_row = np.repeat(np.asarray(targets), group_size)
    strict = np.array([check_completion(int(t), row.tolist(), "strict")[1] for t, row in zip(per_row, tokens)])
    trained_on = np.array([check_completion(int(t), row.tolist(), verifier)[1] for t, row in zip(per_row, tokens)])
    lengths = completion_lengths(tokens)
    by_group = trained_on.reshape(-1, group_size)
    plain = float(np.abs(group_advantages(rewards, group_size, normalise_std=False)).sum())
    scaled = float(np.abs(group_advantages(rewards, group_size)).sum())
    return {
        "reward": float(rewards.mean()),
        "accuracy": float(strict.mean()),                  # always the STRICT verifier: the truth
        "proxy_accuracy": float(trained_on.mean()),        # the verifier the policy was trained against
        "terminated": float((tokens == EOS).any(axis=1).mean()),
        "length": float(lengths.mean()),
        "wrong_tokens": float(lengths[~strict].sum()),     # tokens spent on strictly-wrong completions
        "wrong_count": float((~strict).sum()),
        "zero_var": zero_variance_fraction(rewards, group_size),
        "all_fail": float((~by_group.any(axis=1)).mean()),
        "all_pass": float(by_group.all(axis=1).mean()),
        "kl": stats["kl"],
        "clip_fraction": stats["clip_fraction"],
        # total |advantage| with the std division over total |advantage| without it; nan if every group tied
        "adv_scale": scaled / plain if plain > 1e-12 else float("nan"),
    }


def train_grpo(cfg: GRPOConfig, seed: int, targets: np.ndarray = TRAIN_TARGETS):
    """Train a fresh policy with GRPO. Returns `(params, history)`.

    Uses `init_params`, `copy_params`, `Adam`, `step_rng`, `sample_group`, `group_rewards`,
    `group_advantages`, `make_rollout`, `grpo_update` and `step_metrics`; `history` is the list of
    the per-step metric dicts, one per step, in order.

    Example: `params, history = train_grpo(GRPOConfig(steps=3), seed=0)` gives `len(history) == 3`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_train_grpo() -> None:
    def train_grpo(cfg, seed, targets=TRAIN_TARGETS):         # yours, inside a budget of sampling calls
        with _budget("sample_group", 3 * cfg.steps + 3, mistake=(
                f"train_grpo called sample_group more than {3 * cfg.steps + 3} times for cfg.steps = {cfg.steps}: "
                "does the loop stop after cfg.steps steps, and advance the step each time round?")):
            return globals()["train_grpo"](cfg, seed, targets)

    cfg = GRPOConfig(steps=40)
    params, history = train_grpo(cfg, seed=0)
    assert len(history) == cfg.steps, f"one metrics dict per step: expected {cfg.steps}, got {len(history)}"
    start = init_params(0)
    tokens = sample_group(start, TRAIN_TARGETS, cfg.group_size, step_rng(0, 0))
    expected = float(group_rewards(TRAIN_TARGETS, cfg.group_size, tokens, cfg.verifier).mean())
    assert abs(history[0]["reward"] - expected) < 1e-12, (
        f"step 0's mean reward is {history[0]['reward']:.4f}; sampling the untrained policy with "
        f"step_rng(seed, 0) gives {expected:.4f}. Sample with the step's own rng and BEFORE the first update"
    )
    assert history[0]["kl"] == 0.0 and history[5]["kl"] > 0.0, (
        f"the first batch is sampled from the reference itself (kl {history[0]['kl']}), and later ones from a "
        f"policy that has moved away (step 5: {history[5]['kl']:.4f}). If the KL stays 0, the reference shares "
        "its arrays with the parameters: copy it"
    )
    first = np.mean([h["reward"] for h in history[:5]])
    last = np.mean([h["reward"] for h in history[-5:]])
    assert last > first + 0.2, (
        f"mean reward went from {first:.3f} over the first steps to {last:.3f} over the last: training "
        "should be clearly improving it by now. Check the sign of the update and that each step samples afresh"
    )
    start = init_params(0)
    assert any(not np.array_equal(params[k], start[k]) for k in PARAM_NAMES), "the returned parameters never moved"
    hacked = train_grpo(GRPOConfig(steps=2, verifier="loophole"), seed=0)[1]
    expected = float(group_rewards(TRAIN_TARGETS, cfg.group_size, tokens, "loophole").mean())
    assert abs(hacked[0]["reward"] - expected) < 1e-12, "the reward must come from the verifier the config names (cfg.verifier)"
    small = train_grpo(GRPOConfig(steps=2, group_size=4), seed=0)[1]
    tokens4 = sample_group(start, TRAIN_TARGETS, 4, step_rng(0, 0))
    assert abs(small[0]["length"] - float(completion_lengths(tokens4).mean())) < 1e-12, "cfg.group_size completions per prompt"
    a = train_grpo(GRPOConfig(steps=3, normalise_std=True), seed=0)[0]
    b = train_grpo(GRPOConfig(steps=3, normalise_std=False), seed=0)[0]
    assert any(not np.allclose(a[k], b[k]) for k in PARAM_NAMES), "cfg.normalise_std must reach group_advantages"
    print(f"exercise 10 looks right: mean reward {first:.3f} over the first 5 steps, {last:.3f} over the last 5")


_try("exercise 10", _check_train_grpo, needs=tuple(f"exercise {k}" for k in range(1, 10)))

## 14. The instrument: several seeds, intervals, and a cache

Everything from here on is measured on runs of `train_grpo`, on the same fixed seeds every time (`SEEDS`), and reported as a mean with a
95% interval across seeds (a Student-t interval). One lucky curve proves nothing in
reinforcement learning, because two runs of the SAME code differ; the interval is how big that difference is.
When two settings are compared on the same seeds, the comparison is PAIRED: the difference is taken seed by
seed, so the shared initial policy and the shared first samples cancel out. If a paired interval contains
zero, the lesson says it cannot tell the settings apart, and does not pretend to a winner.

The seeds were fixed before any run, and are not tuned. The cell below is given: the cache, the interval, and
the small functions every later cell uses.

In [ ]:
SEEDS = (0, 1, 2, 3, 4)
BASE = GRPOConfig()
_T95 = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571, 6: 2.447, 7: 2.365, 8: 2.306, 9: 2.262}
_RUNS: dict = {}


def run(cfg: GRPOConfig, seed: int):
    """`(params, series)` for one training run, cached; `series[name]` is an array over steps."""
    if (cfg, seed) not in _RUNS:
        params, history = train_grpo(cfg, seed)
        _RUNS[(cfg, seed)] = (params, {key: np.array([entry[key] for entry in history]) for key in history[0]})
    return _RUNS[(cfg, seed)]


def interval(values) -> tuple[float, float]:
    """Mean and 95% Student-t half-width across seeds."""
    values = np.asarray(values, dtype=float)
    if len(values) < 2:
        return float(values.mean()), float("nan")
    half = _T95[len(values) - 1] * values.std(ddof=1) / np.sqrt(len(values))
    return float(values.mean()), float(half)


def show(values, digits: int = 3) -> str:
    mean, half = interval(values)
    return f"{mean:.{digits}f} ± {half:.{digits}f}"


def block_means(series: np.ndarray, width: int = 10) -> np.ndarray:
    return series[: len(series) // width * width].reshape(-1, width).mean(axis=1)


def steps_to(series: np.ndarray, threshold: float, window: int = 10) -> int:
    """First step at which the trailing `window`-step mean reaches `threshold`; len(series) if never."""
    for s in range(window - 1, len(series)):
        if series[s - window + 1: s + 1].mean() >= threshold:
            return s
    return len(series)


def paired_verdict(a, b, name_a: str, name_b: str, unit: str = "") -> str:
    """Describe b - a over seeds, in words that follow the numbers."""
    diff = np.asarray(b, float) - np.asarray(a, float)
    mean, half = interval(diff)
    if abs(mean) <= half:
        return (f"{name_b} minus {name_a}: {mean:+.3f}{unit} ± {half:.3f}: the interval contains zero, so "
                f"{len(diff)} seeds cannot tell them apart")
    return (f"{name_b} minus {name_a}: {mean:+.3f}{unit} ± {half:.3f}: the interval excludes zero, so "
            f"{name_b} is {'higher' if mean > 0 else 'lower'} across these {len(diff)} seeds")


def paired_word(a, b, higher: str, lower: str, neither: str) -> str:
    """`higher` or `lower` if the paired interval of b - a excludes zero on that side, else `neither`."""
    mean, half = interval(np.asarray(b, float) - np.asarray(a, float))
    return neither if abs(mean) <= half else (higher if mean > 0 else lower)


def _show_baseline() -> None:
    runs = [run(BASE, seed)[1] for seed in SEEDS]
    reward = np.array([block_means(r["reward"]) for r in runs])
    acc = np.array([block_means(r["accuracy"]) for r in runs])
    length = np.array([block_means(r["length"]) for r in runs])
    print(f"GRPO on the toy task: {len(SEEDS)} seeds, {BASE.steps} steps, G={BASE.group_size}, "
          f"{len(TRAIN_TARGETS)} prompts per step ({BASE.group_size * len(TRAIN_TARGETS)} completions), "
          f"reward = {FORMAT_REWARD} for terminating + {CORRECT_REWARD} for being right")
    print("steps        mean reward (95% interval)    accuracy (strict verifier)    mean length")
    for b in range(reward.shape[1]):
        print(f"{10 * b:>3}-{10 * b + 9:<3}   {show(reward[:, b]):<26}    {show(acc[:, b]):<26}    "
              f"{length[:, b].mean():.2f}")
    top = CORRECT_REWARD + FORMAT_REWARD
    print(f"\nthe best possible mean reward is {top}. Untrained, the policy earns {reward[:, 0].mean():.3f}; "
          f"in the last ten steps, {reward[:, -1].mean():.3f} (per seed: "
          f"{', '.join(f'{v:.3f}' for v in reward[:, -1])}).")
    reached = [steps_to(r["accuracy"], 0.9) for r in runs]
    print(f"steps until the 10-step mean accuracy first reaches 0.9, per seed: {reached} "
          f"(mean {np.mean(reached):.1f}; a value of {BASE.steps} would mean never)")


_try("baseline", _show_baseline, needs=tuple(f"exercise {k}" for k in range(1, 11)))

## 15. How often does a group teach nothing?

A group whose completions all earn the same reward has advantage zero everywhere. It costs `G` completions and
contributes nothing (`dapo-zero-advantage`). Early in training every completion in a group can be wrong; late
in training every one can be right. So the fraction of such groups should not be constant, and this task lets
you watch it. The cell below reads it off the baseline runs: the fraction of groups with zero variance (from
your `zero_variance_fraction`), and, separately, the fraction with no correct completion and the fraction
with every completion correct.

One subtlety the table makes visible: a group with NO correct completion is not necessarily a zero-variance
group, because the format reward still separates a completion that terminated from one that did not.

In [ ]:
def _show_zero_variance() -> None:
    runs = [run(BASE, seed)[1] for seed in SEEDS]

    def blocks(key: str) -> np.ndarray:
        return np.array([block_means(r[key]) for r in runs]).mean(axis=0)

    zero, none_right, all_right = blocks("zero_var"), blocks("all_fail"), blocks("all_pass")
    equal_and_wrong = zero - all_right         # zero variance AND nobody right: identical rewards, all wrong
    print("steps     zero-variance   no completion   every completion   ...of which no correct and rewards")
    print("          groups          correct         correct            identical (a zero-variance group)")
    for b in range(len(zero)):
        print(f"{10 * b:>3}-{10 * b + 9:<3}     {zero[b]:.3f}           {none_right[b]:.3f}           "
              f"{all_right[b]:.3f}              {equal_and_wrong[b]:.3f}")
    low = int(zero.argmin())
    print(f"\nzero-variance groups: {zero[0]:.3f} of all groups in steps 0-9, lowest ({zero[low]:.3f}) in steps "
          f"{10 * low}-{10 * low + 9}, and {zero[-1]:.3f} in the last ten steps.")
    print(f"in steps 0-9, {none_right[0]:.3f} of groups had no correct completion, but only "
          f"{equal_and_wrong[0]:.3f} were zero-variance: the format reward "
          f"{'kept the other groups informative' if none_right[0] > equal_and_wrong[0] else 'made no difference'}.")
    print(f"in the last ten steps only {1 - zero[-1]:.3f} of the {len(TRAIN_TARGETS)} groups per step, about "
          f"{(1 - zero[-1]) * len(TRAIN_TARGETS):.1f} of them, still carry any gradient at all.")


_try("zero variance", _show_zero_variance, needs=tuple(f"exercise {k}" for k in range(1, 11)))

The fraction of groups that carry no gradient is the problem DAPO's dynamic sampling addresses: over-sample,
then drop the prompts whose completions are all right or all wrong, so that every prompt in a batch has an
effective gradient (`dapo-dynamic-sampling`). This lesson does not implement it; it measures the problem. With `G = 2` a group has zero variance whenever its two completions tie,
with a large `G` a tie is rare: section 18 measures how that changes with the group size.

## 16. Dividing by the standard deviation

GRPO's advantage divides by the group's standard deviation. Dr. GRPO's critique is that this gives groups
different weights: questions with a small standard deviation, those that are nearly always right or nearly
always wrong, get larger weights (`drgrpo-difficulty-bias`). The first cell measures that weighting with YOUR
`group_advantages` on groups of binary rewards: a group of `G` completions of which `k` are correct, and how
much total advantage each variant hands out. Then it trains with and without the division, on the same seeds,
first with the KL penalty of the base configuration, then with the penalty switched off, and last a control:
no division, with the penalty scaled down by the measured ratio of the two advantage scales.

In [ ]:
def _show_std_weights() -> None:
    g = BASE.group_size
    print(f"one group of G={g} completions, k of them correct (reward 1), the rest wrong (reward 0)")
    print("  k    std of the group   sum |A| with std   sum |A| without std   ratio")
    mass = {}
    for k in range(1, g):
        rewards = np.array([1.0] * k + [0.0] * (g - k))
        with_std = np.abs(group_advantages(rewards, g)).sum()
        without = np.abs(group_advantages(rewards, g, normalise_std=False)).sum()
        mass[k] = (with_std, without)
        print(f"  {k}    {rewards.std():.3f}              {with_std:9.3f}          {without:9.3f}            {with_std / without:.2f}")
    middle = g // 2
    print(f"a group with 1 correct completion hands out {mass[1][0] / mass[middle][0]:.2f} of the total advantage of an evenly "
          f"split group (k={middle}) when the division is on, and {mass[1][1] / mass[middle][1]:.2f} of it when it is off: "
          f"the division lifts the nearly-solved and nearly-hopeless groups relative to the balanced ones")


_try("standard-deviation weights", _show_std_weights, needs=("exercise 5",))


def _show_std_ablation() -> None:
    off = BASE._replace(normalise_std=False)
    on_runs = [run(BASE, seed)[1] for seed in SEEDS]
    off_runs = [run(off, seed)[1] for seed in SEEDS]
    final = lambda rs: [r["accuracy"][-20:].mean() for r in rs]
    speed = lambda rs: [steps_to(r["accuracy"], 0.9) for r in rs]
    calm = lambda rs: [r["zero_var"][-20:].mean() for r in rs]
    print(f"{len(SEEDS)} seeds each, {BASE.steps} steps; accuracy is the strict verifier's, mean of the last 20 steps")
    print(f"                       divide by std        no division")
    print(f"final accuracy         {show(final(on_runs)):<20} {show(final(off_runs))}")
    print(f"steps to 0.9 accuracy  {show(speed(on_runs), 1):<20} {show(speed(off_runs), 1)}")
    print(f"zero-variance groups   {show(calm(on_runs)):<20} {show(calm(off_runs))}   (last 20 steps)")
    print("paired: " + paired_verdict(final(on_runs), final(off_runs), "with std", "without std") + " (final accuracy)")
    print("paired: " + paired_verdict(speed(on_runs), speed(off_runs), "with std", "without std") + " (steps to 0.9)")
    d_final = np.mean(final(off_runs)) - np.mean(final(on_runs))
    per_prompt = {}
    for label, cfg in (("with std", BASE), ("without std", off)):
        accs = []
        for seed in SEEDS:
            tokens = sample_group(run(cfg, seed)[0], TRAIN_TARGETS, 400, np.random.default_rng([SEED, seed, 9]))
            accs.append((group_rewards(TRAIN_TARGETS, 400, tokens) >= CORRECT_REWARD).reshape(len(TRAIN_TARGETS), 400).mean(axis=1))
        per_prompt[label] = np.mean(accs, axis=0)
        print(f"per-prompt accuracy after training, {label}: worst prompt (target {TRAIN_TARGETS[per_prompt[label].argmin()]}) "
              f"{per_prompt[label].min():.3f}, best prompt (target {TRAIN_TARGETS[per_prompt[label].argmax()]}) {per_prompt[label].max():.3f}")
    print(f"so with the KL penalty at beta={BASE.beta}, removing the division "
          + paired_word(speed(on_runs), speed(off_runs), "makes learning slower", "makes learning faster",
                        "makes no difference these seeds can detect") + " to reach 0.9 and "
          + paired_word(final(on_runs), final(off_runs), "raises", "lowers", "does not detectably change")
          + f" the final accuracy ({d_final:+.3f}).")

    # Does the KL penalty matter here? Switch it off and compare again, on the same seeds.
    free_on, free_off = BASE._replace(beta=0.0), BASE._replace(beta=0.0, normalise_std=False)
    on0 = [run(free_on, seed)[1] for seed in SEEDS]
    off0 = [run(free_off, seed)[1] for seed in SEEDS]
    print("\nthe same comparison with the KL penalty switched off (beta=0.0):")
    print(f"                       divide by std        no division")
    print(f"final accuracy         {show(final(on0)):<20} {show(final(off0))}")
    print(f"steps to 0.9 accuracy  {show(speed(on0), 1):<20} {show(speed(off0), 1)}")
    print("paired: " + paired_verdict(final(on0), final(off0), "with std", "without std") + " (final accuracy)")
    print("paired: " + paired_verdict(speed(on0), speed(off0), "with std", "without std") + " (steps to 0.9)")
    e_final = np.mean(final(off0)) - np.mean(final(on0))
    print(f"with beta=0.0, removing the division "
          + paired_word(final(on0), final(off0), "raises", "lowers", "does not detectably change")
          + f" the final accuracy ({e_final:+.3f}) on these {len(SEEDS)} seeds.")

    # A control. Without the division the advantages are smaller (the table above), so the same beta may weigh
    # more against them. Measure how much smaller, divide beta by that, and train without the division again.
    k = float(np.nanmedian(np.concatenate([r["adv_scale"] for r in on_runs])))
    matched = off._replace(beta=BASE.beta / k)
    ctl = [run(matched, seed)[1] for seed in SEEDS]
    print(f"\na control: no division, with beta divided by k = {k:.2f} (the median, over the batches of the runs with the "
          f"division, of total |advantage| with it over total |advantage| without it), so beta={matched.beta:.4f}:")
    print(f"                       divide by std        no division, beta / k")
    print(f"final accuracy         {show(final(on_runs)):<20} {show(final(ctl))}")
    print("paired: " + paired_verdict(final(on_runs), final(ctl), "with std", "without std, beta / k") + " (final accuracy)")
    gap = paired_word(final(on_runs), final(off_runs), "gap", "gap", "") == "gap"
    still = paired_word(final(on_runs), final(ctl), "gap", "gap", "") == "gap"
    if gap and not still:
        print(f"with beta scaled down by k, {len(SEEDS)} seeds no longer detect a difference in final accuracy. So on this task "
              f"the gap at beta={BASE.beta} is accounted for by how strongly one fixed penalty weighs against the smaller "
              "advantages, and no separate effect of the division's re-weighting of groups is left that these seeds can "
              "detect. That is what this control shows and no more: one k for the whole run matches the typical scale of "
              "the advantages, not each group's, and an interval that contains zero is not a proof of no effect.")
    elif gap:
        print(f"the gap in final accuracy survives scaling beta down by k, so the penalty's relative weight does not account "
              f"for it on these {len(SEEDS)} seeds.")
    else:
        print(f"at beta={BASE.beta} these seeds detect no gap in final accuracy, so there is nothing for the control to explain.")
    print("The paper reports its variants ahead of vanilla GRPO at scale (`drgrpo-ablation-result`), in a different setting; "
          "section 21 says what a toy can and cannot say about that.")


_try("standard-deviation ablation", _show_std_ablation, needs=tuple(f"exercise {k}" for k in range(1, 11)))

## 17. Dividing by the length

The published GRPO objective averages each completion's token terms over its own length. For a completion with
a negative advantage that makes each of its tokens weigh less the longer the completion is, so a long wrong
completion is punished less per token than a short wrong one; Dr. GRPO argues this is a bias toward long
wrong answers (`drgrpo-length-bias`), and DAPO argues that sample-level averaging weighs tokens of long
samples too little (`dapo-sample-level-loss`). The first cell shows the weights your `length_scale` gives; the
second measures the length of wrong completions under all three normalisations on the same seeds.

In [ ]:
def _show_length_weights() -> None:
    lengths = np.array([2, 4, LMAX])
    print(f"three wrong completions of lengths {lengths.tolist()}, each with the same negative advantage;")
    print("weight each of their tokens gets, before the advantage and the 1/n average:")
    for mode in ("seq", "const", "batch"):
        print(f"  {mode:<6} {np.round(length_scale(lengths, mode, LMAX), 4).tolist()}")
    ratios = {mode: length_scale(lengths, mode, LMAX) for mode in ("seq", "const", "batch")}
    print(f"a token of the {lengths[0]}-token completion is penalised {ratios['seq'][0] / ratios['seq'][-1]:.1f} times as hard as "
          f"a token of the {lengths[-1]}-token one under 'seq', {ratios['const'][0] / ratios['const'][-1]:.1f} times under 'const', "
          f"{ratios['batch'][0] / ratios['batch'][-1]:.1f} times under 'batch'.")


_try("length weights", _show_length_weights, needs=("exercise 8",))


def _show_length_ablation() -> None:
    modes = ("seq", "const", "batch")
    runs = {m: [run(BASE._replace(length_norm=m), seed)[1] for seed in SEEDS] for m in modes}
    wrong_length = {m: [r["wrong_tokens"].sum() / r["wrong_count"].sum() for r in rs] for m, rs in runs.items()}
    final_length = {m: [r["length"][-20:].mean() for r in rs] for m, rs in runs.items()}
    final_acc = {m: [r["accuracy"][-20:].mean() for r in rs] for m, rs in runs.items()}
    print(f"{len(SEEDS)} seeds each, {BASE.steps} steps")
    print("normalisation   length of wrong completions   mean length, last 20 steps   final accuracy")
    for m in modes:
        print(f"{m:<14}  {show(wrong_length[m], 2):<27}   {show(final_length[m], 2):<26}   {show(final_acc[m])}")
    print("paired: " + paired_verdict(wrong_length["seq"], wrong_length["const"], "'seq'", "'const'", " tokens")
          + " (length of wrong completions)")
    print("paired: " + paired_verdict(wrong_length["seq"], wrong_length["batch"], "'seq'", "'batch'", " tokens")
          + " (length of wrong completions)")
    word = paired_word(wrong_length["const"], wrong_length["seq"], "longer than", "shorter than",
                       "not detectably longer or shorter than")
    print(f"'wrong completions' means every completion the strict verifier rejects, pooled over all {BASE.steps} steps "
          f"of a run. On this task, per-sequence normalisation leaves them {word} "
          f"the constant normalisation does ({np.mean(wrong_length['seq']) - np.mean(wrong_length['const']):+.2f} tokens on average).")


_try("length ablation", _show_length_ablation, needs=tuple(f"exercise {k}" for k in range(1, 11)))

## 18. How big should the group be?

The group is the baseline: `G` completions per prompt give a mean reward estimated from `G` samples, so a
small group is a noisy baseline, and with `G = 2` a group ties (zero variance) whenever its two completions earn
the same reward.
A large group is a better baseline but costs `G` completions per prompt per step. The cell measures four
sizes on the same seeds, and reports the cost in completions as well as in steps, because a step at `G = 16`
is not the same price as a step at `G = 2`.

In [ ]:
def _show_group_size() -> None:
    sizes = (2, 4, BASE.group_size, 16)
    per_step = {g: g * len(TRAIN_TARGETS) for g in sizes}
    runs = {g: [run(BASE._replace(group_size=g), seed)[1] for seed in SEEDS] for g in sizes}
    print(f"{len(SEEDS)} seeds each, {BASE.steps} steps; 'never' means the 10-step mean accuracy did not reach 0.9")
    print("G    completions   steps to 0.9 accuracy    completions to 0.9      final accuracy   early zero-variance")
    means = {}
    for g in sizes:
        steps = [steps_to(r["accuracy"], 0.9) for r in runs[g]]
        never = sum(s >= BASE.steps for s in steps)
        means[g] = np.mean(steps)
        cost = [s * per_step[g] for s in steps]
        print(f"{g:<4} {per_step[g]:>5}/step    {show(steps, 1):<22}   {show(cost, 0):<20}   "
              f"{show([r['accuracy'][-20:].mean() for r in runs[g]]):<14}   "
              f"{np.mean([r['zero_var'][:10].mean() for r in runs[g]]):.3f}"
              + (f"   ({never} of {len(SEEDS)} seeds never reached it)" if never else ""))
    ordered = [means[g] for g in sizes]
    faster = all(a > b for a, b in zip(ordered, ordered[1:]))
    print(f"in steps, learning {'gets faster with every increase in G' if faster else 'does not get faster with every increase in G'}; "
          f"in completions the cheapest of these sizes to reach 0.9 is G={min(sizes, key=lambda g: means[g] * per_step[g])}.")


_try("group size", _show_group_size, needs=tuple(f"exercise {k}" for k in range(1, 11)))

## 19. Reward hacking: a verifier with a loophole

A verifiable reward is only as good as its verifier. DeepSeek-R1-Zero uses rule-based rewards and avoids
neural reward models precisely because the latter are susceptible to reward hacking (`r1-no-neural-rewards`);
a rule-based verifier is not immune, as soon as the rule has a hole. The `"loophole"` verifier from exercise 4
accepts any completion whose digits sum to AT MOST the target, so the empty answer, `EOS` and nothing else,
passes for every prompt.

The policy below is trained on that verifier, same seeds, same everything else. The reward it optimises is the
loophole verifier's. Alongside it the lesson keeps measuring the STRICT verifier's opinion of the same
completions: what the training loop never sees, and what a monitor you trust would report.

In [ ]:
def _show_reward_hack() -> None:
    hacked = BASE._replace(verifier="loophole")
    honest_runs = [run(BASE, seed)[1] for seed in SEEDS]
    hacked_runs = [run(hacked, seed)[1] for seed in SEEDS]
    print(f"{len(SEEDS)} seeds each, {BASE.steps} steps. 'passes' is the pass rate under the verifier the policy trains on;")
    print("'strictly right' is the strict verifier's pass rate on the same completions.")
    print("\ntrained on the LOOPHOLE verifier, by block of steps:")
    print("steps        passes           strictly right   mean length")
    proxy = np.array([block_means(r["proxy_accuracy"]) for r in hacked_runs])
    truth = np.array([block_means(r["accuracy"]) for r in hacked_runs])
    length = np.array([block_means(r["length"]) for r in hacked_runs])
    for b in range(proxy.shape[1]):
        print(f"{10 * b:>3}-{10 * b + 9:<3}     {show(proxy[:, b]):<16} {show(truth[:, b]):<16} {length[:, b].mean():.2f}")
    print("\nlast 20 steps, mean over seeds:")
    print("trained on      passes           strictly right   mean length")
    for name, runs in (("strict", honest_runs), ("loophole", hacked_runs)):
        print(f"{name:<14}  {show([r['proxy_accuracy'][-20:].mean() for r in runs]):<16} "
              f"{show([r['accuracy'][-20:].mean() for r in runs]):<16} "
              f"{show([r['length'][-20:].mean() for r in runs], 2)}")
    gap = [r["proxy_accuracy"][-20:].mean() - r["accuracy"][-20:].mean() for r in hacked_runs]
    reached = [steps_to(r["proxy_accuracy"], 0.9) for r in hacked_runs]
    print(f"\nthe hack: completions the loophole verifier accepts and the strict verifier rejects, {show(gap)} of all "
          f"completions in the last 20 steps; the training reward reached 0.9 in {show(reached, 1)} steps.")
    honest_reward = np.mean([r["proxy_accuracy"][-20:].mean() for r in honest_runs])
    hacked_reward = np.mean([r["proxy_accuracy"][-20:].mean() for r in hacked_runs])
    print(f"what the training reward shows: a run that looks {'better' if hacked_reward > honest_reward else 'no better'} than the "
          f"honest one ({hacked_reward:.3f} against {honest_reward:.3f}). What only the second verifier shows: "
          f"a strict accuracy of {np.mean([r['accuracy'][-20:].mean() for r in hacked_runs]):.3f} and completions of "
          f"{np.mean([r['length'][-20:].mean() for r in hacked_runs]):.2f} tokens on average.")


_try("reward hacking", _show_reward_hack, needs=tuple(f"exercise {k}" for k in range(1, 11)))

Nothing in the training loop is broken: the update did exactly what it was asked to, on the reward it was given.
The two defences are the ones the measurement suggests. Keep an independent check that the policy does not train
against (here, the strict verifier's accuracy, recorded at every step), and read length and content as well as
reward. A curve that reaches its maximum in a handful of steps, with the mean length collapsing at the same
time, is a warning, not a result.

## 20. The KL penalty, measured

`β` sets how strongly the policy is pulled back toward the frozen reference. The reference here is the
untrained policy, so any competent policy is far from it. The first cell trains with three
values of `β` and reports the k3 estimate of the KL over the last 20 steps of a run (each value is the mean of
your `k3_kl` over the sampled tokens of one step). The second checks the estimator itself: on the same
samples, it compares the mean of k3 with the EXACT per-context KL (which is cheap here: the vocabulary has
a handful of tokens) and with the naive estimator `log π − log π_ref` (k1).

In [ ]:
def _show_kl() -> None:
    betas = (0.0, BASE.beta, 0.5)
    runs = {b: [run(BASE._replace(beta=b), seed)[1] for seed in SEEDS] for b in betas}
    print(f"{len(SEEDS)} seeds each, {BASE.steps} steps; last 20 steps")
    print("beta     final accuracy   KL to the reference (k3, per token)   mean length")
    for b in betas:
        print(f"{b:<6}   {show([r['accuracy'][-20:].mean() for r in runs[b]]):<14}   "
              f"{show([r['kl'][-20:].mean() for r in runs[b]]):<36}  "
              f"{show([r['length'][-20:].mean() for r in runs[b]], 2)}")
    kls = [np.mean([r["kl"][-20:].mean() for r in runs[b]]) for b in betas]
    print(f"a larger beta keeps the policy {'closer to' if kls[0] > kls[-1] else 'no closer to'} the reference "
          f"(KL {kls[0]:.3f} at beta={betas[0]}, {kls[-1]:.3f} at beta={betas[-1]}), at a cost in accuracy of "
          f"{np.mean([r['accuracy'][-20:].mean() for r in runs[betas[0]]]) - np.mean([r['accuracy'][-20:].mean() for r in runs[betas[-1]]]):+.3f}.")

    params, _ = run(BASE, SEEDS[0])
    ref = init_params(SEEDS[0])
    tokens = sample_group(params, TRAIN_TARGETS, 400, np.random.default_rng([SEED, SEEDS[0], 3]))
    rows = flatten_completions(TRAIN_TARGETS, 400, tokens)
    logp, logp_ref = token_logprobs(params, rows), token_logprobs(ref, rows)
    k3 = k3_kl(logp, logp_ref)[0]
    k1 = logp - logp_ref
    pi = softmax(policy_logits(params, rows.rem, rows.prev, rows.pos))
    pi_ref = softmax(policy_logits(ref, rows.rem, rows.prev, rows.pos))
    exact = (pi * (np.log(pi) - np.log(pi_ref))).sum(axis=1)
    print(f"\nestimating KL at {len(rows.action)} sampled token rows of the trained policy (seed {SEEDS[0]}):")
    print(f"  exact per-context KL, averaged over the rows: {exact.mean():.4f}")
    print(f"  k3, averaged:                                {k3.mean():.4f}   (per-row spread {k3.std():.3f})")
    print(f"  k1 = log pi - log pi_ref, averaged:          {k1.mean():.4f}   (per-row spread {k1.std():.3f})")
    se = k3.std() / np.sqrt(len(k3))
    print(f"  k3 is never negative ({'true' if (k3 >= 0).all() else 'false'} here); k1 is negative on {(k1 < 0).mean():.3f} of the rows.")
    print(f"  k3 minus the exact KL is {k3.mean() - exact.mean():+.4f}, which is {abs(k3.mean() - exact.mean()) / se:.1f} standard errors "
          f"(treating rows as independent); k3's per-row spread is {k3.std() / k1.std():.2f} times k1's, so at this distance "
          f"from the reference ({exact.mean():.2f} nats per token) k3 is {'noisier' if k3.std() > k1.std() else 'quieter'} than k1.")


_try("KL", _show_kl, needs=tuple(f"exercise {k}" for k in range(1, 11)))

## 21. What this toy does not show

Everything above was measured, and every number depends on this task. Two more measurements bound the claim.
The first: does the trained policy learn the rule, or the prompts it trained on? It never saw the
held-out targets.

In [ ]:
def _show_held_out() -> None:
    train_acc, held_acc = [], []
    for seed in SEEDS:
        params, _ = run(BASE, seed)
        for targets, sink in ((TRAIN_TARGETS, train_acc), (HELD_OUT_TARGETS, held_acc)):
            tokens = sample_group(params, targets, 200, np.random.default_rng([SEED, seed, 4]))
            sink.append(float((group_rewards(targets, 200, tokens) >= CORRECT_REWARD).mean()))
    print(f"strict accuracy of the trained policy ({len(SEEDS)} seeds, 200 samples per prompt)")
    print(f"  on the {len(TRAIN_TARGETS)} prompts it trained on: {show(train_acc)}")
    print(f"  on the {len(HELD_OUT_TARGETS)} prompts it never saw: {show(held_acc)}")
    if np.mean(held_acc) < 0.5 * np.mean(train_acc):
        print("so the policy did not learn the rule 'digits that sum to the target' in any general sense: it has "
              f"{n_parameters(init_params(0))} parameters and {len(TRAIN_TARGETS)} training prompts.")
    else:
        print("so the policy's accuracy carries over to targets it never trained on, at least on these held-out prompts; "
              f"it still has only {n_parameters(init_params(0))} parameters and {len(TRAIN_TARGETS)} training prompts.")


_try("held-out prompts", _show_held_out, needs=tuple(f"exercise {k}" for k in range(1, 11)))

What follows is not measured here and should not be inferred from it:

- **Real reasoning is not shown.** The systems named in section 1 are language models trained on far harder
  tasks; this toy shares neither their scale nor their tasks. It shows the mechanics of the objective and its measurable
  traps; it says nothing about how, or whether, reasoning emerges in a model trained this way.
- **The task is easy by construction.** The policy is given the remaining sum, completions are a few tokens
  long, and one correct answer is one integer equation. Real verifiers grade final answers in open-ended text.
- **The signs of small-scale effects are not the papers' claims.** The critiques cited here are about
  large-model training; this lesson measured what happens on a task with a tiny policy, and where
  its numbers differ from a paper's argument, they answer a different question.
- **KL numbers do not transfer.** The KL here is measured against a randomly initialised reference; it is a
  different object from the KL of a fine-tuned model to the supervised model it started from.
- **A handful of seeds is few.** The intervals are Student-t intervals on a few runs; they say how much these
  runs vary, and they cannot promise the next few will agree.

## 22. Common mistakes

- **Normalising by the wrong thing.** The loss averages over COMPLETIONS. Dividing the summed token weights by
  the number of token rows instead of the number of completions changes the step size and, with `seq`
  normalisation, the balance between short and long completions.
- **Letting the reference move.** If the reference shares arrays with the trained parameters, the KL is
  always zero and the penalty does nothing. Copy it once, before the first update, and never write to it.
- **Forgetting that the advantage is per completion, not per token.** Every token of a completion gets the
  same advantage; look it up through `rows.seq`.
- **Recomputing `logp_old`.** It is the log-probability under the policy that SAMPLED the batch. If you
  recompute it each epoch the ratio is always one and the clip never engages.
- **Trusting a training-reward curve.** Section 19: the reward the policy trains on can rise to its maximum
  while the thing you wanted falls to zero. Measure what you want with something the policy cannot see.
- **Comparing single runs.** Two runs of the same code differ. Compare on the same seeds, pair them, and report
  the interval.

In [ ]:
def _show_tied_group() -> None:
    g = BASE.group_size
    ahead = [CORRECT_REWARD + FORMAT_REWARD] + [FORMAT_REWARD] * (g - 1)
    adv = group_advantages(np.array([FORMAT_REWARD] * g + ahead), g)
    print(f"your group_advantages on a tied group of G={g}: {adv[:g].tolist()}")
    print(f"and on a group in which one completion is ahead of the rest: {np.round(adv[g:], 2).tolist()}")


_try("tied group", _show_tied_group, needs=("exercise 5",))

## 23. Self-check

1. Every completion in a group earns exactly the same reward. With the epsilon in the denominator, what does
   GRPO do with that group?
   - (a) Gives it a very large advantage, because the standard deviation is zero
   - (b) Returns `nan`, so the group has to be dropped by hand
   - (c) Gives every completion advantage exactly zero: the surrogate term contributes no gradient from that group
   - (d) Gives every completion the group's mean reward as its advantage

2. The clipped surrogate is `min(ρ·A, clip(ρ, 1−ε, 1+ε)·A)`. Why a `min` rather than simply clipping the ratio?
   - (a) So that a token whose ratio has already moved the WRONG way, against its advantage, still keeps its full
     gradient and is pushed back, while a token that has moved far enough the RIGHT way stops being pushed
   - (b) So that the objective is always positive
   - (c) So that the gradient is zero whenever the ratio is outside the band, whatever the sign of `A`
   - (d) Because the minimum is faster to compute

3. Two wrong completions in a group share the same negative advantage; one is short and one is long. With
   per-sequence length normalisation, `1/|o_i|`, which token is penalised more, and why does that matter?
   - (a) The token of the long one, so long wrong answers die out
   - (b) Neither: the normalisation only rescales the whole loss
   - (c) The tokens of both equally, because the advantage is the same
   - (d) The token of the short one, so long wrong completions are punished less per token than short ones

4. A policy trained against a buggy verifier reaches the maximum training reward within a few steps. What is
   the right response?
   - (a) Stop: the task is solved
   - (b) Check the same completions against an independent verifier, and read their content and length, before
     trusting the reward
   - (c) Raise the KL coefficient until the reward stops rising
   - (d) Increase the group size

5. The KL penalty is estimated on tokens sampled from the CURRENT policy, with `k3 = exp(u) − u − 1` and
   `u = log π_ref − log π_θ`. Why are the samples drawn from the current policy and not from the reference?
   - (a) Because sampling from the reference is impossible
   - (b) Because the reference has too few tokens
   - (c) Because `KL(π_θ ‖ π_ref)` is an expectation under `π_θ`, so tokens sampled from `π_θ` estimate it without a further correction
   - (d) Because k3 is only defined for the current policy

<details><summary>Answers</summary>

Answers come with this lesson's worked solution when you enrol on Synapsa.
</details>

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")

## What you built

A policy with a hand-derived, finite-difference-checked backward pass; group sampling; a program-checked
verifier with a format reward; the group-relative advantage with its epsilon and its zero-variance case; the
clipped surrogate, the k3 KL penalty and three length normalisations; a GRPO update over several epochs; and a
training loop, all in plain numpy. Then you measured, on a fixed set of seeds with intervals, the learning curve,
how often a group teaches nothing, what the standard-deviation and length normalisations do on this task, how
the group size trades steps for completions, how a verifier with a loophole turns a rising reward into a policy
that stops almost at once, and how far a toy is from the systems it is named after.

The steps you wrote (sample groups, score with a program, normalise within the group, take a clipped update
with a KL penalty to a reference) are the ones the sources cited in section 1 describe. What differs at scale
is the policy and the sampler; this lesson cannot tell you more than that.

In [ ]:
# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    _ALL_CHECKS = (
        ("exercise 1", _check_policy_logits, ()),
        ("exercise 2", _check_logprob_grad, ("exercise 1",)),
        ("exercise 3", _check_sample_group, ("exercise 1",)),
        ("exercise 4", _check_check_completion, ()),
        ("exercise 5", _check_group_advantages, ()),
        ("exercise 6", _check_clipped_surrogate, ()),
        ("exercise 7", _check_k3_kl, ()),
        ("exercise 8", _check_length_scale, ()),
        ("exercise 9", _check_grpo_update,
         ("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 6", "exercise 7", "exercise 8")),
        ("exercise 10", _check_train_grpo, tuple(f"exercise {k}" for k in range(1, 10))),
    )
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in _ALL_CHECKS:
            _try(_name, _check, needs=_needs)
    _progress_board()
    print(f"\nnotebook wall time so far: {time.perf_counter() - _LESSON_T0:.1f}s")
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero. Inside a notebook kernel the
    # board above has already said so, in a line rather than a traceback.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.